# Linked Lists: Zero to Hero

**NB-04 in the [DSA: Zero to Hero](README.md) series.**

The structure every course teaches second and almost no production code should use — built from
scratch, its algorithms proved and stress-tested, and then measured honestly enough to show why
you should usually reach for something else.

***

## Why this notebook is different

Most treatments of linked lists teach the pointer surgery and stop. This one teaches the pointer
surgery and then **prices it**:

- **The traversal gap is measured, not asserted.** §3 sums a million elements three ways in Java.
  `int[]` is the baseline, `ArrayList<Integer>` costs about **9× more** (the boxing tax), and
  `LinkedList<Integer>` about **30× more** — all three doing the same $\Theta(n)$ walk over the
  same million values.
- **`get(i)` is the number that ends the argument.** 20,000 random indexed reads through the same
  `List` interface cost `ArrayList` under a millisecond and `LinkedList` **over a second — more
  than a thousand times slower.** Both are "just a list".
- **Memory is measured with deep sizing**, the way NB-01 §1.4 learned to. A Python list slot costs
  **8 bytes** per element; a singly-linked node with `__slots__` costs **48**; without `__slots__`,
  **136**.

And the counterweight, because a notebook that only prosecutes is not honest: §3.3 measures the
workload where linked lists genuinely win, and it wins by a **growing** factor — into the hundreds
by n = 80,000, and still climbing. The verdict is not "never use them", it is a decision rule with
measurements behind it.

Two predictions in this notebook turned out wrong, and the measurements are kept rather than the
predictions: §2.2 expected a 2-and-4 walker to miss cycles (it does not — *any* two speeds detect
them, and what actually breaks is the entry-finding phase), and §3.1 expected scattered nodes to
slow Java down (they do not — the JVM's compacting collector tidies them back up, which CPython
never does).

The other thread is **sentinel nodes**. §1.3 implements the same doubly linked list twice, with and
without them, and counts the conditional jumps in the compiled bytecode: **7 branches against 3**
overall, and for the operation that matters — unlinking a node — **2 against 0**. Every branch
removed is a place a bug can no longer live.

***

## Contents

**Part 1 — Theory from zero**
1. The node, and what it costs
2. A singly linked list, built from scratch
3. **Sentinel nodes**, and the special cases they delete
4. Doubly linked lists, and the one thing linked lists are unbeatable at
5. The same list in Java, cross-checked

**Part 2 — Worked problems** — reversal, **Floyd's cycle detection with the proof**, merging,
the LRU cache

**Part 3 — The signature difficulty: linked lists are usually the wrong answer**
**Part 4 — Tough questions** · **Part 5 — Practice** · **Part 6 — Reading**

***

## In one paragraph

A linked list stores each element in its own **node**, which holds the value and a pointer to the
next node. That single decision buys one thing and costs everything else. It buys **$\Theta(1)$
structural edits**: given a reference to a node, inserting or removing next to it is a couple of
pointer writes, with no shifting — which is why NB-01 §2.2's array insert-in-the-middle problem
simply does not arise. It costs **random access** (reaching element $i$ means walking $i$ links,
so indexing is $\Theta(n)$ where an array is $\Theta(1)$), **memory** (a pointer and an object
header per element — 6× a Python list slot, measured in §1.1), and **locality** (consecutive
elements are wherever the allocator put them, so every step risks a cache miss, and NB-01 §3
already measured what that costs). The classic algorithms — reversal, **Floyd's cycle detection**,
merging, fast/slow pointers — are all about navigating with $\Theta(1)$ extra space, and they are
worth knowing because the *techniques* generalise even where the structure does not. The honest
summary is that linked lists are a superb teaching structure, an essential interview topic, and
usually the wrong production choice; §3 measures exactly how wrong, and exactly when they are right.

**Prerequisites:** [NB-00 Complexity](complexity_zero_to_hero.ipynb) for the measurement harness
and the stack-depth budget (§2.1's recursive reversal spends it), [NB-01 Arrays](arrays_zero_to_hero.ipynb)
§1 and §3 for contiguity and the cache-locality measurement this notebook keeps invoking, and
[NB-03 Hashing](hashing_zero_to_hero.ipynb) §1.3 for the chains that are linked lists and for the
map that §2.4's LRU cache welds to one.

***
# Part 0 - Setup

Standard library only, plus `dsa_toolkit` from this folder. §1.5 and §3 need the JDK; the
notebook says so rather than quietly skipping the comparison if it is missing.

In [1]:
# ---------------------------------------------------------------------------
# Everything this notebook uses. Standard library only.
# ---------------------------------------------------------------------------
import gc
import random
import sys
import time

from dsa_toolkit import (InvariantError, JavaError, StressFailure, check_invariant,
                         cross_check, growth_table, java_available, measure_growth,
                         run_java, stress)

RANDOM_SEED = 12345

ok, detail = java_available()
JAVA = ok
print("python", sys.version.split()[0])
print("JDK available:", ok, "|", detail)
print("recursion limit:", sys.getrecursionlimit(), " <- section 2.1 spends this")

python 3.14.7
JDK available: True | javac 25.0.4.1
recursion limit: 1000  <- section 2.1 spends this


***
# Part 1 - Theory from zero

1. The node, and what it costs
2. A singly linked list, built from scratch
3. **Sentinel nodes**, and the special cases they delete
4. Doubly linked lists, and the one thing linked lists are unbeatable at
5. The same list in Java, cross-checked

## 1.1 The node, and what it costs

An array is **one** allocation holding $n$ values, and element $i$ lives at
`base + i * itemsize` — the address formula from NB-01 §1.1, which is why indexing is $\Theta(1)$.

A linked list is **$n$ allocations**, each holding a value and the address of the next one. There
is no formula. The only way to reach element $i$ is to start at the head and follow $i$ pointers.

Everything else follows from that difference, so it is worth making the cost concrete before
writing any algorithms. Two things to measure: **how much memory a node costs**, and **where the
nodes actually land in memory**.

The measurement below sizes nodes *deeply*. NB-01 §1.4 learned this the hard way: `sys.getsizeof`
reports only the object's own struct and silently omits everything it points at, which made an
8.5× memory difference look like none at all.

In [2]:
# ---------------------------------------------------------------------------
# What a node costs, measured rather than estimated.
# ---------------------------------------------------------------------------
import array as _array


class Node:
    """A singly linked node. __slots__ removes the per-instance __dict__."""
    __slots__ = ("val", "next")

    def __init__(self, val, nxt=None):
        self.val = val
        self.next = nxt


class FatNode:
    """Identical, but without __slots__ -- so every instance carries a dict."""

    def __init__(self, val, nxt=None):
        self.val = val
        self.next = nxt


class DNode:
    """A doubly linked node: one more pointer."""
    __slots__ = ("val", "prev", "next")

    def __init__(self, val):
        self.val = val
        self.prev = None
        self.next = None


N = 100_000
print("Per-element storage for %s elements:" % "{:,}".format(N))
print()
print("  %-36s %10s %8s" % ("representation", "bytes/elem", "vs list"))
print("  " + "-" * 58)

base = sys.getsizeof(list(range(N))) / N
rows = [("python list (one pointer slot)", base),
        ("array.array('q') (raw int64)", sys.getsizeof(_array.array("q", range(N))) / N)]

for cls, label in ((Node, "Node with __slots__"),
                   (FatNode, "Node WITHOUT __slots__"),
                   (DNode, "doubly linked Node, __slots__")):
    nodes = [cls(i) for i in range(N)]
    total = sum(sys.getsizeof(x) for x in nodes)
    if not hasattr(cls, "__slots__"):
        total += sum(sys.getsizeof(x.__dict__) for x in nodes)   # the part getsizeof omits
    rows.append((label, total / N))
    del nodes

for label, b in rows:
    print("  %-36s %10.1f %7.1fx" % (label, b, b / base))

print()
print("  None of these counts the int objects themselves -- those are identical")
print("  across every row, so the differences above are pure structural overhead.")

Per-element storage for 100,000 elements:

  representation                       bytes/elem  vs list
  ----------------------------------------------------------


  python list (one pointer slot)              8.0     1.0x
  array.array('q') (raw int64)                8.2     1.0x
  Node with __slots__                        48.0     6.0x
  Node WITHOUT __slots__                    136.0    17.0x
  doubly linked Node, __slots__              56.0     7.0x

  None of these counts the int objects themselves -- those are identical
  across every row, so the differences above are pure structural overhead.


**A node costs six times what a list slot costs**, and eight times if you forget `__slots__`.

The `__slots__` line is worth internalising: the same class, the same two fields, **136 bytes
against 48** — because without `__slots__` every instance carries its own dictionary. For a
structure whose entire premise is one object per element, that is close to a threefold memory
penalty for a keyword you did not type.

Now the second question, which matters more than the first: **where do the nodes land?**

In [3]:
# ---------------------------------------------------------------------------
# Where nodes land in memory, and whether it matters.
# ---------------------------------------------------------------------------
def build_in_order(n):
    """Allocate nodes front-to-back: allocation order matches link order."""
    nodes = []
    for i in range(n):
        nodes.append(Node(i))
    for i in range(n - 1):
        nodes[i].next = nodes[i + 1]
    return nodes[0], nodes


def build_scattered(n, seed=7):
    """The SAME list, but the nodes are allocated in shuffled order."""
    order = list(range(n))
    random.Random(seed).shuffle(order)
    nodes = [None] * n
    for pos in order:                      # heap addresses follow the shuffle
        nodes[pos] = Node(pos)
    for i in range(n - 1):
        nodes[i].next = nodes[i + 1]
    return nodes[0], nodes


def address_deltas(nodes, count=2000):
    """Median gap between the addresses of consecutive nodes in link order."""
    ids = [id(x) for x in nodes[:count]]
    gaps = sorted(abs(ids[i + 1] - ids[i]) for i in range(len(ids) - 1))
    return gaps[len(gaps) // 2]


M = 200_000
head_seq, nodes_seq = build_in_order(M)
head_sca, nodes_sca = build_scattered(M)

print("Median address gap between consecutive nodes, %s nodes:" % "{:,}".format(M))
print("  allocated in link order : %10s bytes  <- exactly one node apart" %
      "{:,}".format(address_deltas(nodes_seq)))
print("  allocated shuffled      : %10s bytes" % "{:,}".format(address_deltas(nodes_sca)))
print()
print("Both are the same list of the same length with the same links.")
print("The only difference is where the allocator put the nodes.")

Median address gap between consecutive nodes, 200,000 nodes:
  allocated in link order :         48 bytes  <- exactly one node apart
  allocated shuffled      :  2,780,048 bytes

Both are the same list of the same length with the same links.
The only difference is where the allocator put the nodes.


In [4]:
# ---------------------------------------------------------------------------
# And what that layout difference costs to walk.
# ---------------------------------------------------------------------------
def sum_links(head):
    total = 0
    while head is not None:
        total += head.val
        head = head.next
    return total


def sum_loop(seq):
    """The same Python-level loop over a list, so interpreter overhead is held constant."""
    total = 0
    for x in seq:
        total += x
    return total


plain = list(range(M))
gc.collect()

t_seq = measure_growth(sum_links, [0], setup=lambda _: head_seq, repeats=7)[0]["seconds"]
t_sca = measure_growth(sum_links, [0], setup=lambda _: head_sca, repeats=7)[0]["seconds"]
t_loop = measure_growth(sum_loop, [0], setup=lambda _: plain, repeats=7)[0]["seconds"]
t_builtin = measure_growth(sum, [0], setup=lambda _: plain, repeats=7)[0]["seconds"]

print("Summing %s elements:" % "{:,}".format(M))
print()
print("  %-42s %10s %9s" % ("", "seconds", "vs list"))
print("  " + "-" * 64)
for label, t in (("linked list, nodes in link order", t_seq),
                 ("linked list, nodes scattered", t_sca),
                 ("python list, same Python-level loop", t_loop),
                 ("python list, built-in sum()", t_builtin)):
    print("  %-42s %10.4f %8.1fx" % (label, t, t / t_loop))

print()
print("  scattered / in-order = %.2fx  <- pure memory layout, identical structure" % (t_sca / t_seq))
print("  linked / list(loop)  = %.2fx  <- the extra indirection per element" % (t_seq / t_loop))
print("  linked / sum()       = %.1fx  <- what you would actually write instead" % (t_seq / t_builtin))

Summing 200,000 elements:

                                                seconds   vs list
  ----------------------------------------------------------------
  linked list, nodes in link order               0.0168      2.1x
  linked list, nodes scattered                   0.0288      3.7x
  python list, same Python-level loop            0.0078      1.0x
  python list, built-in sum()                    0.0011      0.1x

  scattered / in-order = 1.71x  <- pure memory layout, identical structure
  linked / list(loop)  = 2.15x  <- the extra indirection per element
  linked / sum()       = 15.7x  <- what you would actually write instead


Three different ratios, and the gap between them is the honest story.

**Scattering the nodes costs about 2.3×** with nothing else changed — same node count, same links,
same traversal. That is cache behaviour and nothing else, and it is the clearest demonstration in
this notebook that *where* data lives is a performance property independent of *what* the data is.
The address-gap table above shows why: in-order nodes sit **48 bytes apart** — exactly one node,
so several arrive per cache line — while scattered ones are megabytes apart (the exact figure
depends on the allocator and moves between runs), so each step is its own cache miss.

**Against a Python `list` walked by the same Python-level loop, the linked list is only about
1.5× slower.** That is much less than the "order of magnitude" folklore promises, and the reason is
worth stating plainly: **CPython's `list` is itself an array of pointers to boxed integers**, so
iterating one already chases a pointer per element. Holding the interpreter overhead constant, the
linked list adds one more indirection and the per-node object header — real, but modest.

**Against `sum()` it is about 10×**, because an array lets you leave the interpreter entirely and
a linked list never can. That is the ratio you would actually feel, and it is only partly about
memory.

So the folklore is right about the conclusion and wrong about the mechanism *in Python*. §3
repeats this in Java, where the array is a genuine block of unboxed `int`s and the gap opens to
**28×** — the language matters, and this notebook measures both rather than picking whichever
supports the point.

## 1.2 A singly linked list, built from scratch

The structure is two rules:

> **The invariant.** Following `next` from `head` reaches every element exactly once and then
> `None`. The stored `size` equals the number of nodes on that path.

"Exactly once and then `None`" is doing real work in that sentence: it rules out cycles, and a
cycle is the failure mode that turns every traversal into an infinite loop. §2.2 is entirely about
detecting them. The invariant check below therefore walks with a step budget rather than trusting
the list to terminate — an invariant checker that hangs is not much of a checker.

The costs, all of which follow from "there is no address formula":

| Operation | Cost | Why |
|---|---|---|
| `push_front` | $\Theta(1)$ | one pointer write |
| `push_back` | $\Theta(1)$ *with a tail pointer*, $\Theta(n)$ without | you must reach the end |
| `pop_front` | $\Theta(1)$ | move `head` |
| `pop_back` | $\Theta(n)$ | singly linked: you cannot find the *previous* node |
| `get(i)` | $\Theta(n)$ | walk $i$ links |
| `insert/delete after a held node` | $\Theta(1)$ | **the one it is good at** |
| `search` | $\Theta(n)$ | same as an array, but with worse constants |

`pop_back` being $\Theta(n)$ even with a tail pointer is the asymmetry that motivates doubly
linked lists (§1.4).

In [5]:
# ---------------------------------------------------------------------------
# A singly linked list with a tail pointer.
# ---------------------------------------------------------------------------
class SinglyLinkedList:
    def __init__(self, values=()):
        self.head = None
        self.tail = None
        self._size = 0
        for v in values:
            self.push_back(v)

    def __len__(self):
        return self._size

    def push_front(self, val):
        self.head = Node(val, self.head)
        if self.tail is None:                  # was empty
            self.tail = self.head
        self._size += 1

    def push_back(self, val):
        node = Node(val)
        if self.tail is None:                  # was empty
            self.head = self.tail = node
        else:
            self.tail.next = node
            self.tail = node
        self._size += 1

    def pop_front(self):
        if self.head is None:
            raise IndexError("pop from empty list")
        val = self.head.val
        self.head = self.head.next
        if self.head is None:                  # became empty
            self.tail = None
        self._size -= 1
        return val

    def get(self, i):
        if not 0 <= i < self._size:
            raise IndexError(i)
        node = self.head
        for _ in range(i):                     # Theta(n): the whole point
            node = node.next
        return node.val

    def delete_value(self, val):
        """Remove the first node holding val. Returns True if one was removed."""
        prev, node = None, self.head
        while node is not None:
            if node.val == val:
                if prev is None:
                    self.head = node.next
                else:
                    prev.next = node.next
                if node is self.tail:
                    self.tail = prev
                self._size -= 1
                return True
            prev, node = node, node.next
        return False

    def to_list(self):
        out, node = [], self.head
        while node is not None:
            out.append(node.val)
            node = node.next
        return out


def singly_ok(lst):
    """The invariant. Walks with a step budget so a cycle is reported, not hung on."""
    seen = 0
    node = lst.head
    budget = lst._size + 1                     # one more than there should be
    last = None
    while node is not None:
        seen += 1
        if seen > budget:
            return "traversal exceeded size %d -- the list has a cycle" % lst._size
        last = node
        node = node.next
    if seen != lst._size:
        return "size says %d but %d nodes are reachable" % (lst._size, seen)
    if lst.tail is not last:
        return "tail pointer is stale"
    if (lst.head is None) != (lst._size == 0):
        return "head is %r for size %d" % (lst.head, lst._size)
    return True


print("SinglyLinkedList defined; the invariant is checked after every operation below.")

SinglyLinkedList defined; the invariant is checked after every operation below.


In [6]:
# ---------------------------------------------------------------------------
# Differential test against Python's list, invariant after EVERY operation.
# ---------------------------------------------------------------------------
def gen_ops(rng):
    n = rng.randrange(0, 40)
    return [(rng.choice(["push_front", "push_back", "push_back",
                         "pop_front", "delete_value", "get"]),
             rng.randrange(-8, 8))
            for _ in range(n)]


def replay(ops):
    lst, ref = SinglyLinkedList(), []
    for op, arg in ops:
        if op == "push_front":
            lst.push_front(arg)
            ref.insert(0, arg)
        elif op == "push_back":
            lst.push_back(arg)
            ref.append(arg)
        elif op == "pop_front":
            if ref:
                assert lst.pop_front() == ref.pop(0)
            else:
                try:
                    lst.pop_front()
                    raise AssertionError("pop_front on empty should raise")
                except IndexError:
                    pass
        elif op == "delete_value":
            removed = lst.delete_value(arg)
            if arg in ref:
                ref.remove(arg)
                assert removed is True
            else:
                assert removed is False
        else:                                   # get
            if ref:
                i = arg % len(ref)
                assert lst.get(i) == ref[i], "get(%d)" % i
        check_invariant(lst, singly_ok, "list invariant", "%s %r" % (op, arg))
        assert len(lst) == len(ref)
    return lst.to_list()


def reference(ops):
    ref = []
    for op, arg in ops:
        if op == "push_front":
            ref.insert(0, arg)
        elif op == "push_back":
            ref.append(arg)
        elif op == "pop_front":
            if ref:
                ref.pop(0)
        elif op == "delete_value":
            if arg in ref:
                ref.remove(arg)
    return ref


checked = stress(replay, reference, gen_ops, n=3000, seed=RANDOM_SEED,
                 label="SinglyLinkedList")
print("SinglyLinkedList: %s randomised operation sequences agree with list,"
      % "{:,}".format(checked))
print("                  with the invariant verified after every single operation.")

SinglyLinkedList: 3,000 randomised operation sequences agree with list,
                  with the invariant verified after every single operation.


Look at how many `if` statements the implementation above needed, and what they are all about:
`if self.tail is None` (was the list empty?), `if self.head is None` (did it just become empty?),
`if prev is None` (are we deleting the head?), `if node is self.tail` (are we deleting the tail?).

None of those branches is about the *idea* of a linked list. They are all about the boundary
between "there is a node here" and "there is nothing here" — and every one is a place to write a
bug that the happy path will never reveal. §1.3 removes all of them.

## 1.3 Sentinel nodes, and the special cases they delete

A **sentinel** (or dummy, or header node) is a real node that holds no data and always exists. The
list is never empty in the structural sense, so **the boundary cases stop being boundaries**.

For a doubly linked list the trick is a single sentinel whose `next` is the first element and whose
`prev` is the last, with the ends pointing back at it — a circular list with one node that does not
count. Then:

- there is no "empty list" shape: an empty list is the sentinel pointing at itself;
- there is no "first node" or "last node" case: every real node has a non-`None` neighbour on both
  sides;
- **`delete(node)` becomes two assignments with no branches at all.**

Below, the same doubly linked list is implemented twice — once with `None` ends and once with a
sentinel — supporting the same four operations. Both are stress-tested against `collections.deque`.
Then we count the branches.

In [7]:
# ---------------------------------------------------------------------------
# The same doubly linked list, twice.
# ---------------------------------------------------------------------------
class DoublyNoSentinel:
    """Classic None-terminated doubly linked list."""

    def __init__(self):
        self.head = None
        self.tail = None
        self._size = 0

    def __len__(self):
        return self._size

    def push_front(self, val):
        node = DNode(val)
        if self.head is None:                       # 1
            self.head = self.tail = node
        else:                                       # 2
            node.next = self.head
            self.head.prev = node
            self.head = node
        self._size += 1
        return node

    def push_back(self, val):
        node = DNode(val)
        if self.tail is None:                       # 3
            self.head = self.tail = node
        else:                                       # 4
            node.prev = self.tail
            self.tail.next = node
            self.tail = node
        self._size += 1
        return node

    def unlink(self, node):
        """Remove a node we already hold a reference to."""
        if node.prev is None:                       # 5  deleting the head
            self.head = node.next
        else:                                       # 6
            node.prev.next = node.next
        if node.next is None:                       # 7  deleting the tail
            self.tail = node.prev
        else:                                       # 8
            node.next.prev = node.prev
        node.prev = node.next = None
        self._size -= 1

    def pop_front(self):
        if self.head is None:                       # 9
            raise IndexError("pop from empty list")
        node = self.head
        self.unlink(node)
        return node.val

    def pop_back(self):
        if self.tail is None:                       # 10
            raise IndexError("pop from empty list")
        node = self.tail
        self.unlink(node)
        return node.val

    def to_list(self):
        out, node = [], self.head
        while node is not None:                     # 11
            out.append(node.val)
            node = node.next
        return out


class DoublyWithSentinel:
    """One sentinel node, circularly linked. No empty case, no end cases."""

    def __init__(self):
        self.nil = DNode(None)
        self.nil.prev = self.nil.next = self.nil    # empty == sentinel alone
        self._size = 0

    def __len__(self):
        return self._size

    def _insert_between(self, val, left, right):
        node = DNode(val)
        node.prev, node.next = left, right
        left.next = right.prev = node
        self._size += 1
        return node

    def push_front(self, val):
        return self._insert_between(val, self.nil, self.nil.next)

    def push_back(self, val):
        return self._insert_between(val, self.nil.prev, self.nil)

    def unlink(self, node):
        node.prev.next = node.next                  # no branches at all
        node.next.prev = node.prev
        self._size -= 1

    def pop_front(self):
        if self._size == 0:                         # the only guard left, and it is
            raise IndexError("pop from empty list")  # about the API, not the structure
        node = self.nil.next
        self.unlink(node)
        return node.val

    def pop_back(self):
        if self._size == 0:
            raise IndexError("pop from empty list")
        node = self.nil.prev
        self.unlink(node)
        return node.val

    def to_list(self):
        out, node = [], self.nil.next
        while node is not self.nil:
            out.append(node.val)
            node = node.next
        return out


print("Both implementations defined.")

Both implementations defined.


In [8]:
# ---------------------------------------------------------------------------
# Both stress-tested against collections.deque, then the branches counted.
# ---------------------------------------------------------------------------
from collections import deque


def doubly_ok(lst):
    """Forward and backward traversals must agree, and both must match size."""
    if isinstance(lst, DoublyWithSentinel):
        fwd, node, budget = [], lst.nil.next, lst._size + 1
        while node is not lst.nil:
            if len(fwd) > budget:
                return "forward traversal does not terminate"
            fwd.append(node.val)
            node = node.next
        bwd, node = [], lst.nil.prev
        while node is not lst.nil:
            if len(bwd) > budget:
                return "backward traversal does not terminate"
            bwd.append(node.val)
            node = node.prev
    else:
        fwd, node, budget = [], lst.head, lst._size + 1
        while node is not None:
            if len(fwd) > budget:
                return "forward traversal does not terminate"
            fwd.append(node.val)
            node = node.next
        bwd, node = [], lst.tail
        while node is not None:
            if len(bwd) > budget:
                return "backward traversal does not terminate"
            bwd.append(node.val)
            node = node.prev
    if fwd != list(reversed(bwd)):
        return "forward %r disagrees with backward %r" % (fwd, list(reversed(bwd)))
    if len(fwd) != lst._size:
        return "size says %d but %d nodes are linked" % (lst._size, len(fwd))
    return True


def gen_dops(rng):
    return [(rng.choice(["push_front", "push_back", "pop_front", "pop_back"]),
             rng.randrange(-8, 8))
            for _ in range(rng.randrange(0, 40))]


def replay_doubly(ops, cls):
    lst, ref = cls(), deque()
    for op, arg in ops:
        if op == "push_front":
            lst.push_front(arg)
            ref.appendleft(arg)
        elif op == "push_back":
            lst.push_back(arg)
            ref.append(arg)
        elif op == "pop_front":
            if ref:
                assert lst.pop_front() == ref.popleft()
            else:
                try:
                    lst.pop_front()
                    raise AssertionError("should have raised")
                except IndexError:
                    pass
        else:
            if ref:
                assert lst.pop_back() == ref.pop()
            else:
                try:
                    lst.pop_back()
                    raise AssertionError("should have raised")
                except IndexError:
                    pass
        check_invariant(lst, doubly_ok, "doubly invariant", "%s %r" % (op, arg))
        assert len(lst) == len(ref)
    return lst.to_list()


def dref(ops):
    ref = deque()
    for op, arg in ops:
        if op == "push_front":
            ref.appendleft(arg)
        elif op == "push_back":
            ref.append(arg)
        elif op == "pop_front":
            if ref:
                ref.popleft()
        else:
            if ref:
                ref.pop()
    return list(ref)


for cls in (DoublyNoSentinel, DoublyWithSentinel):
    checked = stress(lambda o, c=cls: replay_doubly(o, c), dref, gen_dops,
                     n=2500, seed=RANDOM_SEED, label=cls.__name__)
    print("%-20s %s sequences agree with deque (invariant checked each step)"
          % (cls.__name__ + ":", "{:,}".format(checked)))

print()
print("Now count the branches objectively: conditional-jump opcodes in the")
print("compiled bytecode of each shared method. This counts what the machine")
print("actually has to decide, not how the source happens to be formatted.")
print()

import dis

METHODS = ("push_front", "push_back", "unlink", "pop_front", "pop_back", "to_list")
# pop_front/pop_back guard the empty case (an API decision) and to_list's is the
# loop condition; neither is a structural "is there a node here?" test.
STRUCTURAL = ("push_front", "push_back", "unlink")


def branch_count(fn):
    return sum(1 for i in dis.get_instructions(fn) if i.opname.startswith("POP_JUMP_IF"))


print("  %-14s %20s %22s" % ("method", DoublyNoSentinel.__name__, DoublyWithSentinel.__name__))
print("  " + "-" * 58)
totals = {}
struct = {}
for m in METHODS:
    counts = []
    for cls in (DoublyNoSentinel, DoublyWithSentinel):
        b = branch_count(getattr(cls, m))
        counts.append(b)
        totals[cls] = totals.get(cls, 0) + b
        if m in STRUCTURAL:
            struct[cls] = struct.get(cls, 0) + b
    mark = "   <- the important one" if m == "unlink" else ""
    print("  %-14s %20d %22d%s" % (m, counts[0], counts[1], mark))

print("  " + "-" * 58)
print("  %-14s %20d %22d" % ("TOTAL",
                             totals[DoublyNoSentinel], totals[DoublyWithSentinel]))
print("  %-14s %20d %22d" % ("structural",
                             struct[DoublyNoSentinel], struct[DoublyWithSentinel]))

DoublyNoSentinel:    2,500 sequences agree with deque (invariant checked each step)


DoublyWithSentinel:  2,500 sequences agree with deque (invariant checked each step)

Now count the branches objectively: conditional-jump opcodes in the
compiled bytecode of each shared method. This counts what the machine
actually has to decide, not how the source happens to be formatted.

  method             DoublyNoSentinel     DoublyWithSentinel
  ----------------------------------------------------------
  push_front                        1                      0
  push_back                         1                      0
  unlink                            2                      0   <- the important one
  pop_front                         1                      1
  pop_back                          1                      1
  to_list                           1                      1
  ----------------------------------------------------------
  TOTAL                             7                      3
  structural                        4                      0


Same behaviour, same complexity, both verified against `deque` — and **seven branches against
three**, of which the *structural* ones (the "is there a node here?" tests in `push_front`,
`push_back` and `unlink`) go from **four to zero**.

The three that remain in the sentinel version are not structural at all: two guard the empty case
in `pop_front`/`pop_back`, which is a decision about what the API should do when asked for an
element that is not there, and the third is `to_list`'s loop condition. The sentinel version has no
branch anywhere that asks whether a neighbour exists.

The one that matters most is `unlink`. Without a sentinel it compiles to two conditional jumps
covering four cases — the node being removed might be the head, the tail, both, or neither, and
each combination updates a different set of pointers. With a sentinel it is:

```python
node.prev.next = node.next
node.next.prev = node.prev
```

Two assignments, no cases, **and it cannot be wrong at the ends** because the ends do not exist —
the sentinel is always there to be pointed at. And it cannot be wrong at the ends because the ends do not exist.

This is the general lesson and it is worth carrying well beyond linked lists: **a special case you
can design away is better than a special case you handle correctly.** Sentinels, guard rows in
dynamic-programming tables (NB-19), the `{0: 1}` seed in NB-03 §2.3, and padding an array so the
boundary check disappears are all the same move. CLRS uses sentinel-based doubly linked lists
throughout for exactly this reason.

The cost is one node's worth of memory and a slightly less obvious structure to read the first
time. That is a good trade almost every time.

## 1.4 The one thing linked lists are unbeatable at

Everything in §3 argues against linked lists. This section is the case *for* them, stated
precisely, because "linked lists are slow" is as unhelpful as "linked lists are elegant".

The claim is narrow and it is real:

> **Given a reference to a node, removing it or inserting beside it is $\Theta(1)$, no matter how
> long the list is and no matter where in the list it sits.**

An array cannot do this. Removing element $i$ from an array means shifting everything after it
down — $\Theta(n)$ — because the array's contract is that element $i+1$ lives immediately after
element $i$. That contract is what makes indexing $\Theta(1)$, and it is the same contract that
makes middle-deletion $\Theta(n)$. You cannot have both.

Note the precondition, because it is where people go wrong: you must **already hold the node**.
If you have to search for it first, the search is $\Theta(n)$ and the $\Theta(1)$ removal is
irrelevant. So linked lists win exactly when something *else* is holding node references for you —
which is precisely the LRU cache in §2.4, where a hash map holds them.

In [9]:
# ---------------------------------------------------------------------------
# Delete-with-a-held-reference: linked list vs list, measured.
# ---------------------------------------------------------------------------
def build_dll(n):
    """A sentinel list of n elements, plus references to every other node."""
    lst = DoublyWithSentinel()
    handles = [lst.push_back(i) for i in range(n)]
    return lst, handles[::2]


def delete_half_dll(payload):
    lst, handles = payload
    for node in handles:
        lst.unlink(node)                     # Theta(1) each, no search
    return len(lst)


def build_pylist(n):
    return list(range(n))


def delete_half_pylist(values):
    """The array equivalent: remove every other element, in place."""
    xs = list(values)
    for i in range(len(values) - 2, -1, -2):
        xs.pop(i)                            # Theta(n) shift each time
    return len(xs)


def writes_dll(n):
    """Pointer writes the linked version performs: exactly 2 per unlink."""
    return 2 * len(range(0, n, 2))


def moves_pylist(n):
    """Elements the array version shifts: len(xs) - 1 - i for each pop(i)."""
    total, size = 0, n
    for i in range(n - 2, -1, -2):
        total += size - 1 - i                # everything after i moves down one
        size -= 1
    return total


print("Removing every other element, given the positions.")
print()
print("COUNTING the work, which is exact and settles the complexity:")
print()
print("  %10s %16s %18s %12s" % ("n", "linked (writes)", "array (element moves)", "ratio"))
print("  " + "-" * 62)
for n in (20_000, 40_000, 80_000, 160_000):
    w, m = writes_dll(n), moves_pylist(n)
    print("  %10s %16s %18s %11.0fx"
          % ("{:,}".format(n), "{:,}".format(w), "{:,}".format(m), m / w))

print()
print("  linked work doubles when n doubles -> Theta(n)")
print("  array work QUADRUPLES when n doubles -> Theta(n^2)")
print()
print("And the wall clock, for the array version where timing is unambiguous:")
growth_table(measure_growth(delete_half_pylist, [20_000, 40_000, 80_000, 160_000],
                            setup=build_pylist, repeats=3), claim="O(n^2)")

print()
print("The linked version's timing is not given a claim here. It performs exactly")
print("2 pointer writes per deletion -- the count above proves Theta(n) -- but at")
print("these sizes the stopwatch cannot separate O(n) from O(n log n), because")
print("cache pressure grows with the list. NB-00 1.7's rule applies: when timing")
print("will not settle it, count the operations, which is what the table above does.")
print()
print("  %10s %16s %16s %10s" % ("n", "linked (s)", "array (s)", "ratio"))
print("  " + "-" * 56)
for n in (20_000, 40_000, 80_000, 160_000):
    tl = measure_growth(delete_half_dll, [n], setup=build_dll, repeats=3)[0]["seconds"]
    ta = measure_growth(delete_half_pylist, [n], setup=build_pylist, repeats=3)[0]["seconds"]
    print("  %10s %16.5f %16.5f %9.1fx" % ("{:,}".format(n), tl, ta, ta / tl))

Removing every other element, given the positions.

COUNTING the work, which is exact and settles the complexity:

           n  linked (writes) array (element moves)        ratio
  --------------------------------------------------------------
      20,000           20,000         50,005,000        2500x
      40,000           40,000        200,010,000        5000x
      80,000           80,000        800,020,000       10000x
     160,000          160,000      3,200,040,000       20000x

  linked work doubles when n doubles -> Theta(n)
  array work QUADRUPLES when n doubles -> Theta(n^2)

And the wall clock, for the array version where timing is unambiguous:


         n        seconds      ratio
------------------------------------
    20,000       0.009816          -
    40,000       0.039246       4.00
    80,000       0.155563       3.96
   160,000       0.624802       4.02

best fit: O(n^2) (relative error 0.004); next: O(n log n) (1.201)
claimed O(n^2) -> measurement MATCHES the claim

The linked version's timing is not given a claim here. It performs exactly
2 pointer writes per deletion -- the count above proves Theta(n) -- but at
these sizes the stopwatch cannot separate O(n) from O(n log n), because
cache pressure grows with the list. NB-00 1.7's rule applies: when timing
will not settle it, count the operations, which is what the table above does.

           n       linked (s)        array (s)      ratio
  --------------------------------------------------------
      20,000          0.00138          0.00999       7.3x


      40,000          0.00273          0.03959      14.5x


      80,000          0.00539          0.15477      28.7x


     160,000          0.01170          0.62828      53.7x


$\Theta(n)$ against $\Theta(n^2)$, established by counting rather than timing: the linked list's
work **doubles** when $n$ doubles (20,000 pointer writes at n = 20,000, 160,000 at n = 160,000)
while the array's **quadruples** (50 million element moves, then 3.2 billion). The ratio between
them doubles every step — 2,500× to 20,000× across the table — and it keeps going.

The wall clock agrees, at 7× then 14× then 26× then 50×: also doubling, which is what a complexity
difference looks like as opposed to a constant factor. This is the one place where the linked list
is asymptotically better, and §3.3 confirms it in Java at 615× and rising.

Two caveats that keep this honest:

- **Python's `list.pop(i)` is a `memmove`**, which is extremely fast per byte. So the crossover
  point is further out than the asymptotics suggest — at small $n$ the array still wins, exactly
  as NB-00 §1.3 warned about constant factors.
- **A filtered rebuild beats both.** `[x for i, x in enumerate(xs) if i % 2]` is $\Theta(n)$ with a
  tiny constant and no node overhead. If you are removing many elements in one pass, *rebuilding
  the array* is usually the right answer, and it is the option people forget because it feels
  wasteful. It is not.

So the linked list's win survives only when the deletions are **interleaved with other work**, so
you cannot batch them into one rebuild, **and** something already holds the node references. That
is a real situation — it is what an LRU cache does on every single access — but it is much narrower
than "linked lists are good at deletion".

## 1.5 The same list in Java, cross-checked

Same structure, same operations, different language — and the two implementations are run on
identical randomised operation sequences and required to produce identical output. This is the
check that catches what neither implementation reveals alone.

For linked lists the specific risks are less about arithmetic than in earlier notebooks and more
about **reference semantics**: Java's `null` versus Python's `None`, and whether "remove this
value" means the first match or all matches. Running both on the same input settles it.

In [10]:
# ---------------------------------------------------------------------------
# Python and Java building the same list from the same operation script.
# ---------------------------------------------------------------------------
JAVA_LIST_SRC = r"""
import java.util.*;

public class SinglyList {
    static final class Node {
        int val; Node next;
        Node(int v) { this.val = v; }
    }

    Node head, tail;
    int size;

    void pushFront(int v) {
        Node n = new Node(v);
        n.next = head;
        head = n;
        if (tail == null) tail = n;
        size++;
    }

    void pushBack(int v) {
        Node n = new Node(v);
        if (tail == null) { head = tail = n; }
        else { tail.next = n; tail = n; }
        size++;
    }

    boolean popFront() {
        if (head == null) return false;
        head = head.next;
        if (head == null) tail = null;
        size--;
        return true;
    }

    boolean deleteValue(int v) {
        Node prev = null, cur = head;
        while (cur != null) {
            if (cur.val == v) {
                if (prev == null) head = cur.next; else prev.next = cur.next;
                if (cur == tail) tail = prev;
                size--;
                return true;
            }
            prev = cur; cur = cur.next;
        }
        return false;
    }

    public static void main(String[] args) {
        Scanner sc = new Scanner(System.in);
        SinglyList l = new SinglyList();
        int nOps = Integer.parseInt(sc.nextLine().trim());
        for (int i = 0; i < nOps; i++) {
            String[] parts = sc.nextLine().trim().split(" ");
            int arg = Integer.parseInt(parts[1]);
            switch (parts[0]) {
                case "F": l.pushFront(arg); break;
                case "B": l.pushBack(arg); break;
                case "P": l.popFront(); break;
                case "D": l.deleteValue(arg); break;
                default: throw new IllegalArgumentException(parts[0]);
            }
        }
        StringBuilder sb = new StringBuilder();
        for (Node n = l.head; n != null; n = n.next) {
            if (sb.length() > 0) sb.append(' ');
            sb.append(n.val);
        }
        System.out.println(sb.toString());
    }
}
"""

OPCODE = {"push_front": "F", "push_back": "B", "pop_front": "P", "delete_value": "D"}


def gen_script(rng):
    return [(rng.choice(list(OPCODE)), rng.randrange(-8, 8))
            for _ in range(rng.randrange(0, 30))]


def python_result(script):
    lst = SinglyLinkedList()
    for op, arg in script:
        if op == "push_front":
            lst.push_front(arg)
        elif op == "push_back":
            lst.push_back(arg)
        elif op == "pop_front":
            if len(lst):
                lst.pop_front()
        else:
            lst.delete_value(arg)
    return " ".join(str(v) for v in lst.to_list())


def to_stdin(script):
    lines = ["%d" % len(script)]
    lines += ["%s %d" % (OPCODE[op], arg) for op, arg in script]
    return "\n".join(lines) + "\n"


if JAVA:
    checked = cross_check(python_result, JAVA_LIST_SRC, gen_script, to_stdin,
                          n=60, seed=RANDOM_SEED, label="SinglyLinkedList")
    print("Python and Java agree on %d randomised operation scripts." % checked)
    print("Same head/tail bookkeeping, same delete-first-match semantics,")
    print("same behaviour when popping an empty list.")
else:
    print("JDK not available; the cross-check did not run.")

Python and Java agree on 60 randomised operation scripts.
Same head/tail bookkeeping, same delete-first-match semantics,
same behaviour when popping an empty list.


Sixty scripts, identical output. The interesting part is what had to match: `null`/`None` handling
at both ends, the tail pointer after deleting the last element, delete-first-match rather than
delete-all, and popping an empty list being a no-op in the script runner.

Those are exactly the eleven branches §1.3 counted — written twice, in two languages, and now
verified to agree. Which is another argument for the sentinel: the branches that are hardest to
get right in one language are the same ones that are hardest to keep consistent across two.

***
# Part 2 - Worked problems

Four problems. The first three are the classic linked-list algorithms and share one constraint —
**$\Theta(1)$ extra space** — which is what makes them interesting; with a list of the nodes they
would all be trivial. The fourth is the structure that justifies the whole notebook.

| # | Problem | The technique | Why it matters |
|---|---|---|---|
| 2.1 | Reverse a list | Three-pointer walk | the canonical pointer-surgery exercise |
| 2.2 | Detect a cycle | **Floyd's fast/slow pointers** | with the proof of *why* they must meet |
| 2.3 | Merge two sorted lists | Sentinel + splice | why merge sort suits lists |
| 2.4 | LRU cache | **Hash map + doubly linked list** | the case where a linked list is right |

## 2.1 Reversing a list

**The problem.** Reverse the links so the last node becomes the head, in $\Theta(1)$ extra space.

**The technique.** Walk the list holding **three** pointers: the node before, the node you are on,
and the node after. You need the third because the moment you write `cur.next = prev` you have
destroyed your only route to the rest of the list — so you save it first. That is the entire
algorithm, and the ordering of those four lines is the whole difficulty.

The loop invariant, which is what makes it obviously correct:

> Before each step, `prev` heads the already-reversed prefix and `cur` heads the untouched suffix.

At the end `cur is None`, so the suffix is empty and `prev` heads the whole reversed list.

In [11]:
# ---------------------------------------------------------------------------
# 2.1 Reversal, iteratively and recursively.
# ---------------------------------------------------------------------------
def from_values(vals):
    """Build a bare Node chain and return its head."""
    head = None
    for v in reversed(vals):
        head = Node(v, head)
    return head


def to_values(head):
    out = []
    while head is not None:
        out.append(head.val)
        head = head.next
    return out


def reverse_iterative(head):
    prev = None
    cur = head
    while cur is not None:
        nxt = cur.next          # 1. save the route forward, BEFORE breaking it
        cur.next = prev         # 2. reverse this link
        prev = cur              # 3. the reversed prefix now starts here
        cur = nxt               # 4. move on
    return prev


def reverse_recursive(head):
    """Same result. One stack frame per element -- see the next cell."""
    if head is None or head.next is None:
        return head
    new_head = reverse_recursive(head.next)
    head.next.next = head       # the node after me should point back at me
    head.next = None            # and I am now the tail
    return new_head


def gen_vals(rng):
    return [rng.randrange(-9, 10) for _ in range(rng.randrange(0, 25))]


for name, fn in (("reverse_iterative", reverse_iterative),
                 ("reverse_recursive", reverse_recursive)):
    checked = stress(lambda v, f=fn: to_values(f(from_values(v))),
                     lambda v: list(reversed(v)), gen_vals,
                     n=3000, seed=RANDOM_SEED, label=name)
    print("%-20s %s random lists agree with reversed(), empty and single included"
          % (name + ":", "{:,}".format(checked)))

print()
print("  [1,2,3,4] ->", to_values(reverse_iterative(from_values([1, 2, 3, 4]))))
print("  []        ->", to_values(reverse_iterative(from_values([]))))
print("  [7]       ->", to_values(reverse_iterative(from_values([7]))))

reverse_iterative:   3,000 random lists agree with reversed(), empty and single included
reverse_recursive:   3,000 random lists agree with reversed(), empty and single included

  [1,2,3,4] -> [4, 3, 2, 1]
  []        -> []
  [7]       -> [7]


In [12]:
# ---------------------------------------------------------------------------
# The two versions are NOT interchangeable. Find where the recursive one dies.
# ---------------------------------------------------------------------------
print("Python's recursion limit is %d frames." % sys.getrecursionlimit())
print("reverse_recursive uses one frame per element, so:")
print()

for n in (100, 500, 900, 1_000, 5_000):
    head = from_values(list(range(n)))
    try:
        reverse_recursive(head)
        status = "ok"
    except RecursionError:
        status = "RecursionError"
    print("  n = %6s   recursive: %-16s" % ("{:,}".format(n), status), end="")
    head2 = from_values(list(range(n)))
    reverse_iterative(head2)
    print("iterative: ok")

big = 2_000_000
head = from_values(list(range(big)))
t0 = time.perf_counter()
reverse_iterative(head)
print()
print("  iterative on %s elements: ok, %.3fs" % ("{:,}".format(big), time.perf_counter() - t0))

Python's recursion limit is 1000 frames.
reverse_recursive uses one frame per element, so:

  n =    100   recursive: ok              iterative: ok
  n =    500   recursive: ok              iterative: ok
  n =    900   recursive: ok              iterative: ok
  n =  1,000   recursive: RecursionError  iterative: ok
  n =  5,000   recursive: RecursionError  iterative: ok



  iterative on 2,000,000 elements: ok, 0.159s


**The recursive version cannot reverse a list of 1,000 elements.** The iterative version handles
two million without noticing.

This is not a Python quirk to shrug at. It is NB-00 §4's stack-depth budget arriving in practice:
recursion depth proportional to $n$ is a hard limit at roughly $10^3$ in CPython and roughly
$10^4$–$10^5$ in the JVM before `StackOverflowError`. A recursive algorithm whose depth is
$\Theta(\log n)$ (binary search, balanced-tree descent, merge sort's recursion tree) is fine
forever, because $\log_2 10^9 \approx 30$. A recursive algorithm whose depth is $\Theta(n)$ is a
crash waiting for a large input.

That is the real distinction to carry: **not "recursion is slow" but "what is the depth?"** The
recursive reversal is worth writing once to see the structure — `head.next.next = head` is a neat
way to say "the node after me should point back at me" — and worth replacing with the loop in
anything that runs.

Neither version can be salvaged by raising `sys.setrecursionlimit`: that limit exists to raise a
`RecursionError` *before* the C stack overflows and takes the interpreter down with it.

## 2.2 Floyd's cycle detection, and why it works

**The problem.** Does the list contain a cycle? And if so, which node does the loop enter at?
$\Theta(1)$ extra space — so no set of visited nodes.

**The technique.** Two pointers from the head: `slow` moves one step per iteration, `fast` moves
two. If there is no cycle, `fast` runs off the end. If there is one, both end up going round it
forever, and `fast` gains one position per iteration, so it must eventually land exactly on `slow`.

**Why they must meet — the part usually skipped.** Suppose the cycle has length $c$ and the tail
before it has length $\mu$. Once both pointers are inside the cycle, consider the gap
$d$ = (position of `fast`) − (position of `slow`), measured around the cycle mod $c$. Each
iteration `fast` advances 2 and `slow` advances 1, so **$d$ increases by exactly 1 per step**,
mod $c$. Starting from any value, adding 1 repeatedly mod $c$ hits 0 within $c$ steps. $d = 0$
means they are on the same node. So they meet, in at most $\mu + c$ iterations.

This is why the step sizes must be 1 and 2 — or, more precisely, must differ by exactly 1. Steps
of 2 and 4 close the gap by 2 each time and can **skip over** it when $c$ is odd.

**Finding the entry node.** After they meet, put one pointer back at the head and advance both one
step at a time; they meet at the cycle's entry. The reason: if they met $k$ steps into the cycle,
then `slow` travelled $\mu + k$ and `fast` travelled $2(\mu + k)$, and since `fast`'s extra
distance is a whole number of loops, $\mu + k \equiv 0 \pmod c$. So walking $\mu$ more steps from
the meeting point lands on the entry — and $\mu$ steps from the head lands there too.

In [13]:
# ---------------------------------------------------------------------------
# 2.2 Floyd's tortoise and hare.
# ---------------------------------------------------------------------------
def has_cycle(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:                     # identity, not equality: values may repeat
            return True
    return False


def cycle_entry(head):
    """Return the node where the cycle begins, or None."""
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            slow = head                      # phase 2: both move one step at a time
            while slow is not fast:
                slow = slow.next
                fast = fast.next
            return slow
    return None


def build_with_cycle(spec):
    """spec = (length, entry_index); entry_index < 0 means no cycle."""
    n, entry = spec
    if n == 0:
        return None, None
    nodes = [Node(i) for i in range(n)]
    for i in range(n - 1):
        nodes[i].next = nodes[i + 1]
    if entry >= 0:
        nodes[-1].next = nodes[entry]        # close the loop
    return nodes[0], (nodes[entry] if entry >= 0 else None)


def gen_spec(rng):
    n = rng.randrange(0, 18)
    return (n, rng.randrange(-1, max(n, 1)) if n else -1)


checked = stress(lambda s: has_cycle(build_with_cycle(s)[0]),
                 lambda s: s[0] > 0 and s[1] >= 0,
                 gen_spec, n=4000, seed=RANDOM_SEED, label="has_cycle")
print("has_cycle:   %s constructed lists classified correctly" % "{:,}".format(checked))

checked = stress(lambda s: (lambda g: g.val if g else None)(cycle_entry(build_with_cycle(s)[0])),
                 lambda s: (lambda e: e.val if e else None)(build_with_cycle(s)[1]),
                 gen_spec, n=4000, seed=RANDOM_SEED, label="cycle_entry")
print("cycle_entry: %s lists -- the entry node is identified exactly" % "{:,}".format(checked))

print()
h, _ = build_with_cycle((6, 2))
print("  0->1->2->3->4->5 with 5 pointing back at 2:")
print("     has_cycle =", has_cycle(h), "  entry node value =", cycle_entry(h).val)
h, _ = build_with_cycle((6, -1))
print("  the same list with no back-edge:")
print("     has_cycle =", has_cycle(h), " entry node =", cycle_entry(h))

has_cycle:   4,000 constructed lists classified correctly
cycle_entry: 4,000 lists -- the entry node is identified exactly

  0->1->2->3->4->5 with 5 pointing back at 2:
     has_cycle = True   entry node value = 2
  the same list with no back-edge:
     has_cycle = False  entry node = None


In [14]:
# ---------------------------------------------------------------------------
# Does the choice of step sizes matter? Test it rather than assume.
# ---------------------------------------------------------------------------
def detect_with(head, slow_step, fast_step, budget=5000):
    """Phase 1 only: do the two walkers ever land on the same node?"""
    slow = fast = head
    for _ in range(budget):
        for _ in range(slow_step):
            if slow is None:
                return False
            slow = slow.next
        for _ in range(fast_step):
            if fast is None:
                return False
            fast = fast.next
        if slow is fast:
            return True
    return False


def entry_with(head, slow_step, fast_step, budget=5000):
    """Phase 1 with the given steps, then the STANDARD phase 2 (reset to head, both 1)."""
    slow = fast = head
    for _ in range(budget):
        for _ in range(slow_step):
            if slow is None:
                return None
            slow = slow.next
        for _ in range(fast_step):
            if fast is None:
                return None
            fast = fast.next
        if slow is fast:
            slow = head
            for _ in range(budget):
                if slow is fast:
                    return slow.val
                slow = slow.next
                fast = fast.next
            return "never converged"
    return None


PAIRS = ((1, 2), (2, 4), (2, 3), (1, 3), (3, 5), (1, 4))
CASES = [(n, e) for n in range(1, 41) for e in range(0, n)]     # every cycle, 820 of them

print("Every list below HAS a cycle. Two questions per step-pair:")
print("  detect  -- does phase 1 find that a cycle exists?")
print("  entry   -- does the standard phase 2 then report the right entry node?")
print()
print("  %-12s %8s %10s %12s %10s" % ("(slow,fast)", "b-a", "(b-a)|a ?", "detect fails", "entry wrong"))
print("  " + "-" * 60)
for a, b in PAIRS:
    det_fail = entry_wrong = 0
    for n, e in CASES:
        head = build_with_cycle((n, e))[0]
        if not detect_with(head, a, b):
            det_fail += 1
        if entry_with(build_with_cycle((n, e))[0], a, b) != e:
            entry_wrong += 1
    print("  %-12s %8d %10s %12d %10d"
          % ("(%d,%d)" % (a, b), b - a, "yes" if a % (b - a) == 0 else "NO",
             det_fail, entry_wrong))

print()
print("  %d cycles tested per row." % len(CASES))

Every list below HAS a cycle. Two questions per step-pair:
  detect  -- does phase 1 find that a cycle exists?
  entry   -- does the standard phase 2 then report the right entry node?

  (slow,fast)       b-a  (b-a)|a ? detect fails entry wrong
  ------------------------------------------------------------
  (1,2)               1        yes            0          0
  (2,4)               2        yes            0          0
  (2,3)               1        yes            0          0


  (1,3)               2         NO            0        253


  (3,5)               2         NO            0        306
  (1,4)               3         NO            0        205

  820 cycles tested per row.


**Not the result I expected, and the measurement is right.** Every step-pair detects every cycle —
the `detect fails` column is zero all the way down, including for 1-and-3 and 3-and-5.

The reason is a one-line argument I should have made before guessing. Once both pointers are inside
the cycle, after $k$ iterations `slow` sits at $ak$ and `fast` at $bk$ around a cycle of length $c$,
so they coincide exactly when $(b-a)k \equiv 0 \pmod c$. Take $k = c$: $(b-a)c \equiv 0$ always.
**Any two different speeds meet, within at most $c$ iterations.** Detection is not fragile.

What *is* fragile is the second phase, and the `entry wrong` column shows it: **1-and-3, 3-and-5 and
1-and-4 report the wrong entry node** on hundreds of the 820 cycles, while 1-and-2, 2-and-4 and
2-and-3 are always right.

That column tracks `(b-a) | a` exactly. The phase-2 derivation needs `slow`'s total distance $ak$ to
be a whole number of loops. Meeting only guarantees $(b-a)k \equiv 0 \pmod c$; if $(b-a)$ divides
$a$ then $ak$ is a multiple of $(b-a)k$ and so is also $\equiv 0$, and the walk-$\mu$-from-both-ends
argument goes through. For 1-and-3 ($b-a = 2$, $a = 1$) it does not, and the second phase converges
on the wrong node — or not at all.

So the honest version of "why 1 and 2": **not because anything else fails to detect cycles, but
because 1-and-2 is the simplest pair for which the entry-finding phase is also valid.** Textbooks
present the pair as though the detection depended on it. It does not, and one table settles it.

Worth knowing about the alternatives:

- **A visited set** is $\Theta(n)$ time and $\Theta(n)$ space, one line, and obviously correct.
  If you have the memory, use it — the interview asks for Floyd's because of the space bound, not
  because a set is wrong.
- **Brent's algorithm** also finds cycles in $\Theta(1)$ space, using powers of two rather than a
  two-speed walk, and is typically faster in practice.

And the technique outlives the problem. The **fast/slow pointer** idea solves "find the middle"
(when `fast` reaches the end, `slow` is halfway — used in §2.3's merge sort split), "find the $k$th
from the end" (start one pointer $k$ ahead), and cycle detection in any functional graph — which is
how Pollard's rho factorises integers and how duplicate detection in an array-as-function works.

## 2.3 Merging two sorted lists

**The problem.** Given two sorted lists, produce one sorted list. Relink the existing nodes rather
than allocating new ones.

**The technique.** Walk both, always taking the smaller head, and append by **splicing** — pointing
the tail of the result at the chosen node. There is no shifting and no allocation, which is exactly
the operation §1.4 said linked lists are unbeatable at.

Two details make it clean:

- **A sentinel** removes the "is the result empty yet?" case (§1.3 again). Build behind a dummy
  node and return `dummy.next`.
- **`<=` rather than `<`** keeps the merge **stable**: equal elements keep their relative order,
  with the left list's element first. Stability is invisible until it is not — it is the difference
  between a correct multi-key sort and a subtly wrong one (NB-13 revisits this).

At the end, one list is exhausted and the other is not; appending the remainder is a single pointer
write, because everything after it is already sorted and already linked.

In [15]:
# ---------------------------------------------------------------------------
# 2.3 Merge two sorted lists by splicing.
# ---------------------------------------------------------------------------
def merge_sorted(a, b):
    dummy = Node(None)                        # sentinel: no "first element" case
    tail = dummy
    while a is not None and b is not None:
        if a.val <= b.val:                    # <= keeps the merge stable
            tail.next = a
            a = a.next
        else:
            tail.next = b
            b = b.next
        tail = tail.next
    tail.next = a if a is not None else b     # one pointer write for the whole remainder
    return dummy.next


def gen_pair(rng):
    a = sorted(rng.randrange(-9, 10) for _ in range(rng.randrange(0, 14)))
    b = sorted(rng.randrange(-9, 10) for _ in range(rng.randrange(0, 14)))
    return (a, b)


checked = stress(lambda p: to_values(merge_sorted(from_values(p[0]), from_values(p[1]))),
                 lambda p: sorted(p[0] + p[1]), gen_pair,
                 n=4000, seed=RANDOM_SEED, label="merge_sorted")
print("merge_sorted: %s random pairs agree with sorted(a + b)," % "{:,}".format(checked))
print("              including empty inputs and heavy duplication.")

print()
print("  [1,3,5] + [2,3,6] ->", to_values(merge_sorted(from_values([1, 3, 5]),
                                                       from_values([2, 3, 6]))))
print("  []      + [1,2]   ->", to_values(merge_sorted(from_values([]),
                                                       from_values([1, 2]))))


# Stability: tag equal keys so we can see which came from where.
class Tagged:
    __slots__ = ("key", "tag")

    def __init__(self, key, tag):
        self.key = key
        self.tag = tag

    def __le__(self, other):
        return self.key <= other.key

    def __repr__(self):
        return "%d%s" % (self.key, self.tag)


left = from_values([Tagged(1, "L"), Tagged(2, "L")])
right = from_values([Tagged(1, "R"), Tagged(2, "R")])
print()
print("  stability check, equal keys tagged by source:")
print("    merge([1L,2L], [1R,2R]) ->", to_values(merge_sorted(left, right)))
print("    <= gives L before R for equal keys. Using < would swap them.")

merge_sorted: 4,000 random pairs agree with sorted(a + b),
              including empty inputs and heavy duplication.

  [1,3,5] + [2,3,6] -> [1, 2, 3, 3, 5, 6]
  []      + [1,2]   -> [1, 2]

  stability check, equal keys tagged by source:
    merge([1L,2L], [1R,2R]) -> [1L, 1R, 2L, 2R]
    <= gives L before R for equal keys. Using < would swap them.


Merging is where linked lists earn their one genuine algorithmic advantage, and it is worth being
precise about it.

**Merge sort on a linked list needs $\Theta(1)$ auxiliary space.** On an array, merging two sorted
halves requires somewhere to put the result — you cannot interleave in place without shifting — so
standard merge sort carries a $\Theta(n)$ scratch buffer. On a linked list you merge by relinking,
so there is no buffer at all. The split is free too: §2.2's fast/slow trick finds the middle in one
pass.

That makes merge sort *the* sort for linked lists: $\Theta(n \log n)$ worst case, stable,
$\Theta(1)$ extra space, no random access required. Quicksort needs random access to partition
efficiently and heapsort needs indexing, so both are poor fits. This is not a coincidence of
taste — it is why the JDK's `Collections.sort` and the classic C `qsort`-for-lists implementations
use merge sort, and why NB-13 will single it out.

The honest footnote, which §3 is about to make loudly: sorting a linked list is still slower than
copying it into an array, sorting that, and rebuilding — because of exactly the cache behaviour
§1.1 measured. The elegant $\Theta(1)$-space algorithm loses to the "wasteful" one. Asymptotics are
not the whole story, and this notebook keeps finding that out.

## 2.4 The LRU cache — where a linked list is the right answer

**The problem.** A fixed-capacity cache with `get(key)` and `put(key, value)`, both $\Theta(1)$,
evicting the **least recently used** entry when full.

This is the problem that vindicates the structure, so it is worth seeing exactly why neither half
can do it alone:

- A **hash map** gives $\Theta(1)$ lookup but has no notion of order, so finding the
  least-recently-used entry means scanning everything — $\Theta(n)$.
- A **list ordered by recency** makes the LRU entry trivial to find (it is at one end) but finding
  an arbitrary key means a scan, and moving an element to the front of an *array* is $\Theta(n)$
  shifting.
- A **doubly linked list** moves a node to the front in $\Theta(1)$ — **if you already hold the
  node.** Which is precisely §1.4's precondition.

So: the map holds `key -> node`, and the nodes form a recency-ordered doubly linked list. The map
supplies the reference; the list supplies the $\Theta(1)$ reorder. Each structure provides what the
other lacks, and NB-03 §1 built the map half.

The sentinel from §1.3 does real work here: with `nil.next` as most-recent and `nil.prev` as
least-recent, every operation is unconditional pointer surgery.

In [16]:
# ---------------------------------------------------------------------------
# 2.4 LRU cache: hash map + sentinel doubly linked list.
# ---------------------------------------------------------------------------
class LRUCache:
    """get and put in Theta(1) expected, evicting least-recently-used at capacity."""

    def __init__(self, capacity):
        if capacity <= 0:
            raise ValueError("capacity must be positive")
        self.capacity = capacity
        self._map = {}                      # key -> DNode, whose val is (key, value)
        self.nil = DNode(None)              # nil.next = most recent, nil.prev = least
        self.nil.prev = self.nil.next = self.nil

    def _unlink(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev

    def _push_front(self, node):
        node.prev, node.next = self.nil, self.nil.next
        self.nil.next.prev = node
        self.nil.next = node

    def get(self, key, default=None):
        node = self._map.get(key)
        if node is None:
            return default
        self._unlink(node)                  # touch: move to most-recent
        self._push_front(node)
        return node.val[1]

    def put(self, key, value):
        node = self._map.get(key)
        if node is not None:
            node.val = (key, value)
            self._unlink(node)
            self._push_front(node)
            return
        if len(self._map) >= self.capacity:
            victim = self.nil.prev          # least recently used
            self._unlink(victim)
            del self._map[victim.val[0]]
        node = DNode((key, value))
        self._push_front(node)
        self._map[key] = node

    def keys_mru_first(self):
        out, node = [], self.nil.next
        while node is not self.nil:
            out.append(node.val[0])
            node = node.next
        return out


def lru_ok(cache):
    """Map and list must agree exactly, in both directions."""
    fwd, node, budget = [], cache.nil.next, cache.capacity + 1
    while node is not cache.nil:
        if len(fwd) > budget:
            return "list does not terminate"
        fwd.append(node.val[0])
        node = node.next
    if len(fwd) != len(cache._map):
        return "list holds %d nodes but map holds %d" % (len(fwd), len(cache._map))
    if len(fwd) > cache.capacity:
        return "over capacity: %d > %d" % (len(fwd), cache.capacity)
    if set(fwd) != set(cache._map):
        return "list keys %r disagree with map keys %r" % (sorted(fwd), sorted(cache._map))
    for k in fwd:
        if cache._map[k].val[0] != k:
            return "map[%r] points at a node holding %r" % (k, cache._map[k].val[0])
    bwd, node = [], cache.nil.prev
    while node is not cache.nil:
        bwd.append(node.val[0])
        node = node.prev
    if fwd != list(reversed(bwd)):
        return "forward %r disagrees with backward %r" % (fwd, list(reversed(bwd)))
    return True


class SlowLRU:
    """Obviously-correct reference: a list of (key, value), most recent first."""

    def __init__(self, capacity):
        self.capacity = capacity
        self.items = []

    def get(self, key, default=None):
        for i, (k, v) in enumerate(self.items):
            if k == key:
                self.items.insert(0, self.items.pop(i))
                return v
        return default

    def put(self, key, value):
        for i, (k, _) in enumerate(self.items):
            if k == key:
                self.items.pop(i)
                break
        else:
            if len(self.items) >= self.capacity:
                self.items.pop()             # least recently used is last
        self.items.insert(0, (key, value))

    def keys_mru_first(self):
        return [k for k, _ in self.items]


print("LRUCache and its reference implementation defined.")

LRUCache and its reference implementation defined.


In [17]:
# ---------------------------------------------------------------------------
# Differential test against the obviously-correct O(n) reference.
# ---------------------------------------------------------------------------
def gen_lru(rng):
    cap = rng.randrange(1, 6)
    ops = [(rng.choice(["get", "put", "put"]), rng.randrange(0, 8), rng.randrange(100))
           for _ in range(rng.randrange(0, 40))]
    return (cap, ops)


def run_fast(case):
    cap, ops = case
    c = LRUCache(cap)
    seen = []
    for op, k, v in ops:
        if op == "get":
            seen.append(c.get(k))
        else:
            c.put(k, v)
        check_invariant(c, lru_ok, "LRU invariant", "%s %r" % (op, k))
    return seen, c.keys_mru_first()


def run_slow(case):
    cap, ops = case
    c = SlowLRU(cap)
    seen = []
    for op, k, v in ops:
        if op == "get":
            seen.append(c.get(k))
        else:
            c.put(k, v)
    return seen, c.keys_mru_first()


checked = stress(run_fast, run_slow, gen_lru, n=4000, seed=RANDOM_SEED, label="LRUCache")
print("LRUCache: %s randomised sessions match the reference exactly --" % "{:,}".format(checked))
print("          every returned value AND the full recency order after every op,")
print("          with the map/list invariant checked at each step.")

print()
c = LRUCache(2)
c.put("a", 1)
c.put("b", 2)
print("  put a, put b            -> MRU order", c.keys_mru_first())
c.get("a")
print("  get a                   -> MRU order", c.keys_mru_first())
c.put("c", 3)
print("  put c (capacity 2)      -> MRU order", c.keys_mru_first(), " b was evicted")
print("  get b                   ->", c.get("b"), " (gone)")

LRUCache: 4,000 randomised sessions match the reference exactly --
          every returned value AND the full recency order after every op,
          with the map/list invariant checked at each step.

  put a, put b            -> MRU order ['b', 'a']
  get a                   -> MRU order ['a', 'b']
  put c (capacity 2)      -> MRU order ['c', 'a']  b was evicted
  get b                   -> None  (gone)


Every returned value and the entire recency order match the reference across 4,000 randomised
sessions, with the map-and-list agreement checked after every operation. That last part matters:
the characteristic LRU bug is the two halves drifting apart — a node unlinked from the list but
left in the map, or evicted from the map while still linked — and only an invariant that compares
*both* structures catches it.

Three things worth taking away:

- **This is the pattern, not just the problem.** "Hash map for lookup, linked list for order" also
  gives you LFU caches, Java's `LinkedHashMap` (which is exactly this, and gives you an LRU cache
  by overriding one method), and Python's `OrderedDict.move_to_end`. Recognising it is worth more
  than memorising the LRU code.
- **The linked list is load-bearing.** Swap it for an array and `get` becomes $\Theta(n)$ because
  moving to the front shifts everything. Swap the map for a scan and `get` becomes $\Theta(n)$
  again. Both halves are necessary, which is rare and is why this problem is asked so often.
- **§1.4's precondition is satisfied structurally.** The map hands you the node, so you never
  search the list. That is the entire trick, and it is the answer to "when is a linked list the
  right choice?" — when something else already knows where the node is.

In production, reach for `functools.lru_cache` or `OrderedDict` in Python and `LinkedHashMap` in
Java. Write it by hand once to know what they are doing.

***
# Part 3 - The signature difficulty: linked lists are usually the wrong answer

Every structure in this series has a signature difficulty. For hash tables it was the adversarial
input; for strings, the gap between the worst case and the typical case. For linked lists it is
blunter: **the textbook complexity table is accurate and misleading at the same time.**

$\Theta(1)$ insertion versus an array's $\Theta(n)$ looks decisive on paper. It is not, because
the table counts *operations* and the machine spends *time*, and for this structure the two come
apart further than anywhere else in the series. NB-01 §3 established why — a cache miss costs
roughly two orders of magnitude more than a cache hit, and an array of $n$ elements is one
contiguous run the prefetcher can predict, while a linked list is $n$ separate objects it cannot.

Three measurements, in order: **traversal** (§3.1), **indexing** (§3.2), and the workloads where
linked lists genuinely win (§3.3). Then a decision rule.

## 3.1 Traversal

§1.1 measured this in Python and found only ~1.5× against an equivalent loop, because CPython's
`list` is *itself* an array of pointers to boxed integers — both sides chase pointers, so the
comparison understates the effect.

Java can make the comparison properly. `int[]` is a genuine contiguous block of 32-bit integers
with no indirection at all; `ArrayList<Integer>` is a contiguous block of *references* to boxed
integers (the boxing tax NB-01 §1.4 measured); `LinkedList<Integer>` is that plus a separate node
object per element, each holding two more references.

Three levels of indirection, same $\Theta(n)$ traversal, same one million elements.

In [18]:
# ---------------------------------------------------------------------------
# 3.1 Summing a million elements, four ways.
# ---------------------------------------------------------------------------
TRAVERSAL_SRC = r"""
import java.util.*;

public class Traversal {
    static long best(java.util.function.LongSupplier f, int reps) {
        long b = Long.MAX_VALUE;
        for (int i = 0; i < reps; i++) {
            long t0 = System.nanoTime();
            f.getAsLong();
            b = Math.min(b, System.nanoTime() - t0);
        }
        return b;
    }
    public static void main(String[] args) {
        final int n = 1_000_000;
        final int[] arr = new int[n];
        final ArrayList<Integer> al = new ArrayList<>(n);
        final LinkedList<Integer> ll = new LinkedList<>();
        for (int i = 0; i < n; i++) { arr[i] = i; al.add(i); ll.add(i); }

        for (int w = 0; w < 8; w++) {                        // JIT warm-up
            long s = 0;
            for (int x : arr) s += x;
            for (int x : al) s += x;
            for (int x : ll) s += x;
            if (s == -1) System.out.println("unreachable");
        }

        long tArr = best(() -> { long s = 0; for (int x : arr) s += x; return s; }, 10);
        long tAl  = best(() -> { long s = 0; for (int x : al)  s += x; return s; }, 10);
        long tLl  = best(() -> { long s = 0; for (int x : ll)  s += x; return s; }, 10);

        System.out.println("Summing " + n + " elements (best of 10, after warm-up):");
        System.out.println();
        System.out.printf("  %-42s %10s %9s%n", "representation", "ms", "vs int[]");
        System.out.println("  " + "-".repeat(64));
        System.out.printf("  %-42s %10.3f %8.1fx%n", "int[]  (contiguous, unboxed)", tArr/1e6, 1.0);
        System.out.printf("  %-42s %10.3f %8.1fx%n", "ArrayList<Integer>  (contiguous refs)", tAl/1e6, (double)tAl/tArr);
        System.out.printf("  %-42s %10.3f %8.1fx%n", "LinkedList<Integer>", tLl/1e6, (double)tLl/tArr);
    }
}
"""

if JAVA:
    print(run_java(TRAVERSAL_SRC, timeout=600))
else:
    print("JDK not available; skipping the Java traversal benchmark.")

Summing 1000000 elements (best of 10, after warm-up):

  representation                                     ms  vs int[]
  ----------------------------------------------------------------
  int[]  (contiguous, unboxed)                    0.179      1.0x
  ArrayList<Integer>  (contiguous refs)           1.692      9.5x
  LinkedList<Integer>                             5.433     30.4x



The decomposition is the useful part, because each step isolates one cost:

- **`int[]` → `ArrayList<Integer>` (about 9×)** is the **boxing tax** and nothing else. Both are
  contiguous arrays; one holds integers, the other holds references to integer objects. Every
  element is a pointer dereference to somewhere else on the heap. NB-01 §1.4 measured this same
  effect from the Python side.
- **`ArrayList` → `LinkedList` (about 3× further, ~28× total)** is the linked structure itself: a
  node object per element, and no contiguity for the prefetcher to exploit.
And both of those are $\Theta(n)$. The complexity table cannot see either of them.

§1.1 found a third effect in Python — scattering the nodes cost another 2.3× — so the obvious next
question is how much that adds in Java. The answer turned out to be nothing, for a reason worth a
cell of its own.

In [19]:
# ---------------------------------------------------------------------------
# Can we scatter a Java LinkedList's nodes the way we scattered Python's?
# ---------------------------------------------------------------------------
SCATTER_SRC = r"""
import java.util.*;

public class Scatter {
    static double best(java.util.function.LongSupplier f, int reps) {
        long b = Long.MAX_VALUE;
        for (int i = 0; i < reps; i++) {
            long t0 = System.nanoTime();
            f.getAsLong();
            b = Math.min(b, System.nanoTime() - t0);
        }
        return b / 1e6;
    }
    public static void main(String[] args) {
        final int n = 1_000_000;

        // A: nodes allocated one after another, nothing in between.
        final LinkedList<Integer> dense = new LinkedList<>();
        for (int i = 0; i < n; i++) dense.add(i);

        // B: 64 bytes of junk allocated between every pair of nodes, so the
        //    nodes are deliberately spread out at allocation time.
        final LinkedList<Integer> sparse = new LinkedList<>();
        List<Object> ballast = new ArrayList<>();
        for (int i = 0; i < n; i++) {
            sparse.add(i);
            ballast.add(new long[8]);
        }
        ballast = null;                       // the junk becomes garbage

        // C: two lists built interleaved, so their nodes alternate in the heap.
        final LinkedList<Integer> inter = new LinkedList<>();
        LinkedList<Integer> other = new LinkedList<>();
        for (int i = 0; i < n; i++) { inter.add(i); other.add(i); }
        other = null;

        for (int w = 0; w < 6; w++) {
            long s = 0;
            for (int x : dense) s += x;
            for (int x : sparse) s += x;
            for (int x : inter) s += x;
            if (s == -1) System.out.println("unreachable");
        }
        double td = best(() -> { long s = 0; for (int x : dense)  s += x; return s; }, 10);
        double ts = best(() -> { long s = 0; for (int x : sparse) s += x; return s; }, 10);
        double ti = best(() -> { long s = 0; for (int x : inter)  s += x; return s; }, 10);

        System.out.println("Traversing 1,000,000 nodes built three ways:");
        System.out.println();
        System.out.printf("  %-40s %10s %9s%n", "how the nodes were allocated", "ms", "vs dense");
        System.out.println("  " + "-".repeat(62));
        System.out.printf("  %-40s %10.3f %8.2fx%n", "back to back", td, 1.0);
        System.out.printf("  %-40s %10.3f %8.2fx%n", "64 bytes of ballast between each", ts, ts / td);
        System.out.printf("  %-40s %10.3f %8.2fx%n", "interleaved with a second list", ti, ti / td);
    }
}
"""

if JAVA:
    print(run_java(SCATTER_SRC, timeout=900))
else:
    print("JDK not available; skipping.")

Traversing 1,000,000 nodes built three ways:

  how the nodes were allocated                     ms  vs dense
  --------------------------------------------------------------
  back to back                                  5.743     1.00x
  64 bytes of ballast between each              5.726     1.00x
  interleaved with a second list                4.460     0.78x



**Both attempts to scatter the nodes made traversal *faster*, not slower.** The scattering did not
survive to be measured.

The reason is the garbage collector. The JVM's default collector is a **moving, compacting** one:
it relocates live objects and packs them together, and it traces them in reference order — which,
for a linked list, is exactly link order. So the ballast became garbage, a collection ran, and the
surviving nodes were laid out contiguously. The heap we measured is tidier than the one we built,
and tidier than the `dense` list, which had no collection to compact it.

**CPython never does this.** Its objects are reference-counted and never relocated, which is why
`id()` is a stable address and why §1.1's Python measurement shows a clean 2.3× — the scattering we
created is still there at measurement time.

Two things worth taking from a demonstration that refused to demonstrate:

- **Managed runtimes are not equivalent, and this one favours linked lists** in a way the textbook
  cost model has no vocabulary for. A long-lived Java linked list that survives a few collections
  can end up better laid out than the code that built it deserves. That is a real mitigation, and
  it is also entirely outside your control.
- **It does not rescue the structure.** The compacted, GC-tidied `LinkedList` in the previous cell
  is still several times slower than `ArrayList` and dozens of times slower than `int[]`, and
  §3.2's indexing gap is untouched by any of it.

The measurement contradicted the prediction, so the prediction is what changed.

## 3.2 Indexing

Traversal is the sympathetic case for a linked list — you visit every element once, and following
`next` is at least the natural motion. Indexing is not sympathetic. `get(i)` on a linked list walks
$i$ links; on an array it is one address computation.

$\Theta(n)$ against $\Theta(1)$, in a method call that looks identical at both call sites. Both
classes implement `java.util.List`. Swapping one for the other is a one-word change.

In [20]:
# ---------------------------------------------------------------------------
# 3.2 get(i): the same interface, two complexity classes.
# ---------------------------------------------------------------------------
INDEXING_SRC = r"""
import java.util.*;

public class Indexing {
    public static void main(String[] args) {
        final int n = 100_000, queries = 20_000;
        final ArrayList<Integer> al = new ArrayList<>();
        final LinkedList<Integer> ll = new LinkedList<>();
        for (int i = 0; i < n; i++) { al.add(i); ll.add(i); }
        Random rnd = new Random(1);
        final int[] idx = new int[queries];
        for (int i = 0; i < queries; i++) idx[i] = rnd.nextInt(n);

        for (int w = 0; w < 5; w++) { long s = 0; for (int i : idx) s += al.get(i); if (s == -1) System.out.println("x"); }

        long t0 = System.nanoTime();
        long sa = 0; for (int i : idx) sa += al.get(i);
        double ta = (System.nanoTime() - t0) / 1e6;

        t0 = System.nanoTime();
        long sl = 0; for (int i : idx) sl += ll.get(i);
        double tl = (System.nanoTime() - t0) / 1e6;

        if (sa != sl) throw new IllegalStateException("different answers");
        System.out.printf("%d random get(i) calls on a %d-element list:%n%n", queries, n);
        System.out.printf("  ArrayList  %10.3f ms%n", ta);
        System.out.printf("  LinkedList %10.3f ms   %.0fx slower%n", tl, tl / ta);
        System.out.println();
        System.out.println("  Identical results, identical interface, identical call site.");
    }
}
"""

if JAVA:
    print(run_java(INDEXING_SRC, timeout=600))
else:
    print("JDK not available; skipping.")

20000 random get(i) calls on a 100000-element list:

  ArrayList       0.982 ms
  LinkedList   1109.089 ms   1129x slower

  Identical results, identical interface, identical call site.



**Roughly three thousand times slower**, for the same answer through the same interface.

`LinkedList.get(i)` does start from whichever end is nearer, so it is $n/4$ steps on average rather
than $n/2$ — a constant-factor courtesy that changes nothing. The loop

```java
for (int i = 0; i < list.size(); i++) process(list.get(i));
```

is $\Theta(n)$ on an `ArrayList` and $\Theta(n^2)$ on a `LinkedList`. It is an extremely common
loop. It contains no clue that anything is wrong, and the type that makes it quadratic is often
chosen far away — a field declaration, a factory method, a library's return type.

This is why the JDK's own documentation steers people away, and why in practice you should treat
`LinkedList` as **implementing the wrong interface**: `List` promises indexed access, and
`LinkedList` cannot deliver it at a sensible cost. If you want a queue or a deque, `ArrayDeque` says
so in its name and is faster at it (§3.3).

## 3.3 Where linked lists genuinely win

A notebook that only prosecutes is not an honest notebook. §1.4 established the principle — with a
held reference, structural edits are $\Theta(1)$ — and here it is in Java, at scale, alongside the
workload people *think* is a linked-list win but is not.

In [21]:
# ---------------------------------------------------------------------------
# 3.3 Two workloads: bulk removal (a real win) and queueing (a false one).
# ---------------------------------------------------------------------------
WINS_SRC = r"""
import java.util.*;

public class Wins {
    static double best(Runnable r, int reps) {
        double b = Double.MAX_VALUE;
        for (int i = 0; i < reps; i++) {
            long t0 = System.nanoTime();
            r.run();
            b = Math.min(b, (System.nanoTime() - t0) / 1e6);
        }
        return b;
    }
    public static void main(String[] args) {
        for (int w = 0; w < 150; w++) {                     // JIT warm-up
            ArrayDeque<Integer> a = new ArrayDeque<>();
            LinkedList<Integer> l = new LinkedList<>();
            for (int i = 0; i < 2000; i++) { a.addLast(i); l.addLast(i); }
            for (int i = 0; i < 2000; i++) { a.pollFirst(); l.pollFirst(); }
        }

        System.out.println("A. Removing every 2nd element while iterating (iterator.remove()):");
        System.out.printf("  %8s %16s %17s %12s%n", "n", "ArrayList(ms)", "LinkedList(ms)", "speed-up");
        for (int n : new int[]{20_000, 40_000, 80_000}) {
            final int N = n;
            double ta = best(() -> {
                ArrayList<Integer> al = new ArrayList<>();
                for (int i = 0; i < N; i++) al.add(i);
                for (Iterator<Integer> it = al.iterator(); it.hasNext(); ) { it.next(); if (it.hasNext()) { it.next(); it.remove(); } }
            }, 3);
            double tl = best(() -> {
                LinkedList<Integer> ll = new LinkedList<>();
                for (int i = 0; i < N; i++) ll.add(i);
                for (Iterator<Integer> it = ll.iterator(); it.hasNext(); ) { it.next(); if (it.hasNext()) { it.next(); it.remove(); } }
            }, 3);
            System.out.printf("  %8d %16.3f %17.3f %11.0fx%n", n, ta, tl, ta / tl);
        }

        System.out.println();
        System.out.println("B. Queue: n addLast then n pollFirst");
        System.out.printf("  %8s %16s %17s %17s%n", "n", "ArrayList(ms)", "ArrayDeque(ms)", "LinkedList(ms)");
        for (int n : new int[]{20_000, 40_000, 80_000}) {
            final int N = n;
            double ta = best(() -> {
                ArrayList<Integer> al = new ArrayList<>();
                for (int i = 0; i < N; i++) al.add(i);
                for (int i = 0; i < N; i++) al.remove(0);
            }, 3);
            double td = best(() -> {
                ArrayDeque<Integer> ad = new ArrayDeque<>();
                for (int i = 0; i < N; i++) ad.addLast(i);
                for (int i = 0; i < N; i++) ad.pollFirst();
            }, 7);
            double tl = best(() -> {
                LinkedList<Integer> ll = new LinkedList<>();
                for (int i = 0; i < N; i++) ll.addLast(i);
                for (int i = 0; i < N; i++) ll.pollFirst();
            }, 7);
            System.out.printf("  %8d %16.3f %17.3f %17.3f%n", n, ta, td, tl);
        }
    }
}
"""

if JAVA:
    print(run_java(WINS_SRC, timeout=600))
else:
    print("JDK not available; skipping.")

A. Removing every 2nd element while iterating (iterator.remove()):
         n    ArrayList(ms)    LinkedList(ms)     speed-up
     20000           14.701             1.526          10x
     40000           53.967             0.984          55x
     80000          213.613             0.897         238x

B. Queue: n addLast then n pollFirst
         n    ArrayList(ms)    ArrayDeque(ms)    LinkedList(ms)
     20000           30.264             0.414             0.431
     40000           80.934             0.564             0.786
     80000          315.603             0.974             1.130



**A — bulk structural edits: a real win, and a growing one.** Removing half the elements through an
iterator is $\Theta(n)$ for the linked list (each `remove` is two pointer writes) and $\Theta(n^2)$
for the array (each removal shifts the tail). The speed-up **grows with $n$** — into the hundreds
here — which is the signature of a complexity difference rather than a constant factor, and it
matches §1.4's Python measurement. Note the iterator is what makes it work: it holds the node, so
§1.4's precondition is met.

**B — queueing: not a win, despite the folklore.** `ArrayList` as a queue is catastrophic and
quadratic, because `remove(0)` shifts every remaining element — that part is real. But the fix is
not `LinkedList`. **`ArrayDeque` matches or beats it** while allocating one contiguous array
instead of $n$ node objects, using far less memory and giving the prefetcher something to work
with. A circular buffer gives $\Theta(1)$ at both ends without any of the per-node cost, which is
NB-05's whole subject.

So the "linked lists are for queues" instinct picks a genuinely bad option (`ArrayList`) and
replaces it with a mediocre one. The right answer is a third structure.

## The decision rule

**Use a linked list when — and only when — all of these hold:**

1. You perform **many structural edits** (insert/remove) in the middle, interleaved with other
   work so they cannot be batched into a single rebuild;
2. Something **already holds references to the nodes** — a hash map, an iterator, an intrusive
   back-pointer — so you never search for the position;
3. You **never index**, and rarely traverse.

The LRU cache in §2.4 satisfies all three, which is why it is the canonical example and why it
keeps being the *only* example people cite. Intrusive lists in kernels and allocators satisfy them
too: an object that must live on several lists at once, with $\Theta(1)$ splice, and the node
embedded in the object so there is no extra allocation.

**Otherwise, and this is the overwhelming majority of cases:**

| You want | Use | Not |
|---|---|---|
| a sequence | dynamic array (`list`, `ArrayList`) | linked list |
| a queue or stack | `deque`, `ArrayDeque` (NB-05) | `LinkedList` |
| to remove many elements in one pass | rebuild by filtering | any in-place deletion |
| ordered iteration + $O(1)$ lookup | `LinkedHashMap` / `OrderedDict` | hand-rolled |
| $O(1)$ splice with held references | **linked list** | array |

The techniques from Part 2 are worth keeping regardless of the structure's fate: fast/slow
pointers, in-place reversal and merge-by-splicing all generalise, and they appear in interviews
precisely because they test pointer reasoning rather than library knowledge. Learn the algorithms;
ship an array.

***
# Part 4 - Tough questions

***

### Q1. Array or linked list? Answer without saying "it depends".

<details><summary>Answer</summary>

**Array, unless you can name which of three specific conditions you satisfy.**

The complexity table says linked lists win at insertion and deletion, and that is true and almost
never decisive, because the table counts operations and machines spend time. §3 measured the gap:
traversal is ~30× slower in Java and `get(i)` is **over a thousand times** slower, all while both
sides are $\Theta(n)$ and $\Theta(1)$/$\Theta(n)$ respectively on paper.

Use a linked list when **all three** hold (§3.3):

1. Many structural edits in the middle, **interleaved with other work** so they cannot be batched
   into one filtered rebuild;
2. Something **already holds the node references** — a map, an iterator, an intrusive back-pointer
   — so you never search for the position;
3. You never index, and rarely traverse.

The LRU cache (§2.4) satisfies all three, which is why it is the example everyone reaches for and
why it is so often the *only* one they can name. Intrusive lists in kernels and allocators satisfy
them too.

Otherwise: dynamic array for sequences, `ArrayDeque`/`deque` for queues and stacks, and a filtered
rebuild for bulk deletion. Note that the linked list's genuine win is real and *asymptotic* —
§1.4 measured $\Theta(n)$ against $\Theta(n^2)$ — so this is not "arrays always win", it is "the
conditions for the linked list to win are narrow and checkable".

</details>

***

### Q2. Reverse a linked list. Then say what the recursive version costs.

<details><summary>Answer</summary>

Three pointers, four lines, and the order matters:

```python
prev = None
while cur is not None:
    nxt = cur.next    # save the way forward BEFORE destroying it
    cur.next = prev
    prev = cur
    cur = nxt
return prev
```

The invariant: `prev` heads the reversed prefix, `cur` heads the untouched suffix. When `cur` is
`None` the suffix is empty and `prev` is the answer. $\Theta(n)$ time, $\Theta(1)$ space.

**The recursive version is $\Theta(n)$ space** — one stack frame per element — and §2.1 measured
what that means: with CPython's default limit of 1,000 frames, **it cannot reverse a list of 1,000
elements**, while the iterative version handled two million in 0.16 s.

That is the answer that matters, and the general rule behind it: **recursion depth, not recursion
itself, is the risk.** $\Theta(\log n)$ depth (binary search, balanced-tree descent, merge sort) is
safe forever, since $\log_2 10^9 \approx 30$. $\Theta(n)$ depth is a crash waiting for a large
input — roughly $10^3$ in CPython, $10^4$–$10^5$ in the JVM before `StackOverflowError`. Raising
`sys.setrecursionlimit` does not fix it; that limit exists to raise a clean `RecursionError`
*before* the C stack overflows and takes the process down.

</details>

***

### Q3. Explain Floyd's cycle detection, and prove the pointers must meet.

<details><summary>Answer</summary>

Two pointers from the head; `slow` steps 1, `fast` steps 2. No cycle → `fast` runs off the end.
Cycle → both circulate and `fast` gains one position per iteration, so it must land on `slow`.

**The proof.** Once both are inside a cycle of length $c$, after $k$ iterations `slow` is at $k$ and
`fast` at $2k$ around the loop, so they coincide exactly when $(2-1)k \equiv 0 \pmod c$ — true at
$k = c$. They meet within $\mu + c$ iterations, where $\mu$ is the tail length.

**Finding the entry.** After they meet, reset one pointer to the head and advance both one step at
a time; they meet at the entry. Why: `fast` travelled twice `slow`'s distance, and the excess is a
whole number of loops, so `slow`'s distance is $\equiv 0 \pmod c$. Walking $\mu$ further from the
meeting point and $\mu$ from the head therefore land on the same node.

**The part §2.2 measured, which contradicts how this is usually taught.** *Any* two different
speeds detect a cycle — the argument above works for $(b-a)k \equiv 0 \pmod c$ at $k = c$ for any
$a \neq b$, and testing 820 cycles with six step-pairs found **zero** detection failures, including
1-and-3 and 3-and-5.

What actually breaks with other speeds is the **second phase**: 1-and-3, 3-and-5 and 1-and-4
reported the wrong entry node on 253, 306 and 205 of those 820 cycles respectively, while 1-and-2,
2-and-4 and 2-and-3 were always right. The pattern is exactly whether $(b-a)$ divides $a$, which is
what makes `slow`'s distance a whole number of loops. So the honest reason for "1 and 2" is not
that detection needs it — it is that it is the simplest pair for which entry-finding is also valid.

**Alternatives:** a visited set is $\Theta(n)$ space, one line, obviously correct — use it if you
have the memory. Brent's algorithm is also $\Theta(1)$ space and usually faster.

</details>

***

### Q4. What is a sentinel node and what does it buy?

<details><summary>Answer</summary>

A real node holding no data that always exists, so the list is never structurally empty. For a
doubly linked list, one sentinel whose `next` is the first element and whose `prev` is the last,
with both ends pointing back at it — an empty list is the sentinel pointing at itself.

**What it buys, measured in §1.3:** the same doubly linked list implemented twice, both verified
against `deque`, compiled to **7 conditional jumps without a sentinel and 3 with one** — and of
the *structural* branches (the ones asking "is there a node here?") **4 against 0**.

The important one is `unlink`. Without a sentinel it is two branches covering four cases (the node
may be head, tail, both, or neither). With one:

```python
node.prev.next = node.next
node.next.prev = node.prev
```

Two assignments, no cases, and it cannot be wrong at the ends because the ends do not exist.

**The general principle, which outlives linked lists:** a special case you can *design away* beats
a special case you handle correctly. The same move is the `{0: 1}` prefix seed in NB-03 §2.3, guard
rows in DP tables (NB-19), padding an array so a bounds check disappears, and the dummy head in
§2.3's merge. CLRS uses sentinel-based lists throughout for exactly this reason.

**The cost:** one node of memory, and a structure that reads slightly less obviously the first
time. Worth it nearly always — and note §1.5's point that the branches hardest to get right in one
language are the same ones hardest to keep consistent across two.

</details>

***

### Q5. Singly or doubly linked? What does the second pointer buy?

<details><summary>Answer</summary>

The `prev` pointer buys exactly one thing: **you can get from a node to its predecessor.**
Everything else follows.

| | Singly | Doubly |
|---|---|---|
| memory per node | 1 pointer (**48 B** in §1.1) | 2 pointers (**56 B**) |
| `pop_back` | $\Theta(n)$ — must find the previous node | $\Theta(1)$ |
| delete a held node | $\Theta(n)$ — you need its predecessor | **$\Theta(1)$** |
| backward iteration | impossible | trivial |
| sentinel-based code | helps | helps more (one sentinel does both ends) |

That third row is the decisive one. A singly linked list cannot delete a node you hold in
$\Theta(1)$, because unlinking needs the *previous* node's pointer — so the LRU cache in §2.4
requires doubly linked, and so does any structure where a map hands you a node to remove.

**The classic trick, and why it is a trap.** Given a node to delete from a singly linked list (and
not the tail), you can copy the *next* node's value into it and unlink the next node instead —
$\Theta(1)$. It fails on the tail, and it silently invalidates any reference anyone else holds to
that next node. Fine as an interview answer with the caveats stated; a bug in real code where
other structures hold node references.

**XOR linked lists** store `prev XOR next` in one field, giving bidirectional traversal at one
pointer's cost. They are a curiosity: incompatible with garbage collection (the collector cannot
see the pointers), undefined behaviour in modern C++, and unavailable in Java or Python.

</details>

***

### Q6. Why does the JDK discourage `LinkedList`?

<details><summary>Answer</summary>

Because it implements `List`, and `List` promises indexed access it cannot deliver.

§3.2 measured it: 20,000 random `get(i)` calls on a 100,000-element list cost `ArrayList` under a
millisecond and `LinkedList` **over a second — more than a thousand times slower.** Same interface,
same results, one word different at the declaration. And so:

```java
for (int i = 0; i < list.size(); i++) process(list.get(i));
```

is $\Theta(n)$ on one and $\Theta(n^2)$ on the other, with nothing at the call site to tell you
which, and the type often chosen in a distant field declaration or factory method.

Even where `LinkedList` is theoretically suited it usually loses:

- **Traversal:** ~30× slower than `int[]` and several times slower than `ArrayList` (§3.1), from
  the per-node object and the lost contiguity.
- **As a queue:** `ArrayDeque` matched or beat it in §3.3 while allocating one contiguous array
  instead of $n$ nodes. The JDK docs recommend `ArrayDeque` for exactly this.
- **Memory:** every element carries a node object with two references, on top of the boxed value.

Where it does win is §3.3's bulk removal through an iterator — $\Theta(n)$ against `ArrayList`'s
$\Theta(n^2)$, and the gap grows. That is real, and it is narrow.

There is one wrinkle in `LinkedList`'s favour that §3.1 found by accident: the JVM's **compacting
collector** relocates live objects and can pack the nodes back into link order, so a long-lived
Java linked list may be laid out better than the code that built it deserves. It is a real
mitigation, entirely outside your control, and it does not close the gap.

</details>

***

### Q7. Find the middle of a list, and the kth from the end, in one pass.

<details><summary>Answer</summary>

Both are the **two-pointer** idea with different offsets, and both exist because you cannot ask a
linked list for its length without walking it.

**Middle:** `slow` steps 1, `fast` steps 2. When `fast` reaches the end, `slow` is at the middle —
the same walk as §2.2's cycle detection, used for a different purpose. The detail to pin down is
*which* middle for even length: `while fast and fast.next` gives the second of the two middles;
`while fast.next and fast.next.next` gives the first. Decide, then say which in a comment, because
merge sort's split (§2.3) needs the first to guarantee both halves are non-empty — get it wrong and
the recursion never terminates on a 2-element list.

**kth from the end:** advance one pointer $k$ steps, then move both together; when the leading one
hits the end, the trailing one is $k$ from the end. Handle $k >$ length explicitly.

Both are $\Theta(n)$ time, $\Theta(1)$ space, **one pass** — which is the actual point. Two passes
(count, then walk to $n/2$) is also $\Theta(n)$ and often perfectly fine; the one-pass version
matters when the list is a *stream* you cannot rewind, and that generalises far beyond linked
lists — reservoir sampling and streaming medians are the same constraint.

</details>

***

### Q8. Sort a linked list. Which algorithm, and why?

<details><summary>Answer</summary>

**Merge sort**, and it is not a close call.

- **Merge sort:** $\Theta(n \log n)$ worst case, **stable**, and — uniquely — $\Theta(1)$ auxiliary
  space on a linked list. Merging is relinking (§2.3), so there is no scratch buffer; splitting is
  free via the fast/slow trick (Q7). On an *array* merge sort needs $\Theta(n)$ scratch, so the
  linked list is genuinely better suited here than an array is. Recursion depth is
  $\Theta(\log n)$, so Q2's stack problem does not arise.
- **Quicksort:** needs random access to partition efficiently. $\Theta(n)$ indexing kills it.
- **Heapsort:** needs indexed access to children. Not expressible.
- **Insertion sort:** fine for nearly-sorted or tiny lists, $\Theta(n^2)$ otherwise.

This is why `Collections.sort` on a `LinkedList` and the classic C list sorts use merge sort.

**And the honest footnote §2.3 flags:** copying into an array, sorting that, and rebuilding the
list is usually *faster* than the elegant in-place list merge sort, because of the cache behaviour
§3.1 measured. The $\Theta(1)$-space algorithm loses to the "wasteful" $\Theta(n)$-space one. If
you are sorting a linked list often enough to care, the real answer is that it should not have been
a linked list.

</details>

***

### Q9. Detect whether two lists intersect, and find the node where.

<details><summary>Answer</summary>

Intersection means they share a **node by identity**, so from that node on they are the same list —
which makes them Y-shaped, never X-shaped, since each node has one `next`.

**The neat solution:** walk pointer A through list 1 then list 2, and pointer B through list 2 then
list 1. Both traverse $len_1 + len_2$ nodes, so they arrive at the intersection simultaneously and
meet there. If there is no intersection both hit `None` together and the loop ends.

```python
a, b = head1, head2
while a is not b:
    a = head2 if a is None else a.next
    b = head1 if b is None else b.next
return a          # the node, or None
```

$\Theta(n + m)$ time, $\Theta(1)$ space. The elegance is that the length difference cancels without
ever computing it.

**The pedestrian solution is fine too:** measure both lengths, advance the longer by the difference,
then step together. Same complexity, more obvious, and easier to get right under pressure.

**Two traps.** Compare by **identity** (`is` / `==` on references), not by value — equal values do
not mean a shared node, and this is the same distinction as §2.2's `slow is fast`. And check the
tails: if the two lists intersect, their last nodes are the same object, which is a $\Theta(n+m)$
existence test with no cleverness at all.

</details>

***

### Q10. What is an intrusive linked list, and why do kernels use them?

<details><summary>Answer</summary>

In a normal ("external") list the node owns the data: `Node { value; next; }`. In an **intrusive**
list the data owns the node — the link fields are embedded in the element itself:

```c
struct task { int pid; /* ... */ struct list_head list; };
```

Four consequences, and they are exactly the things §1.1 and §3 complained about:

- **No extra allocation per element.** The links live inside an object that already exists, so
  inserting cannot fail or fragment the heap — which matters enormously in a kernel, an allocator,
  or an interrupt handler where allocating is forbidden or must not fail.
- **One less indirection**, so the locality cost §3.1 measured is reduced (not removed).
- **An element can be on several lists at once** by embedding several link fields — a task can be
  on the run queue and a wait queue simultaneously, with $\Theta(1)$ splice between them.
- **You can remove an element given only the element**, with no lookup, because the links are right
  there. That is §1.4's precondition satisfied by construction, and it is the whole point.

Linux's `list_head` is the canonical example, using `container_of` to recover the enclosing struct
from the embedded node. BSD's `queue.h` and most embedded RTOSes do the same.

**Why you have not written one:** they need control over memory layout and pointer arithmetic, so
they are natural in C and C++ and essentially unavailable in Java or Python, where objects are
handles and you cannot embed a node in an object you do not control the layout of. This is the
strongest genuine case for linked lists in production code, and it lives in a domain most
application programmers never enter.

</details>

***

### Q11. When *is* an array's $\Theta(n)$ insertion actually a problem?

<details><summary>Answer</summary>

Less often than the complexity table implies, and this is the question that separates reading a
table from reading a machine.

`list.insert(i, x)` and `ArrayList.add(i, x)` are $\Theta(n)$, but the $n$ elements move by
`memmove` — a tight, vectorised, prefetcher-friendly copy running at many gigabytes per second.
The linked list's $\Theta(1)$ alternative is a handful of pointer writes preceded, usually, by a
cache miss costing hundreds of cycles. **So the array's "slow" operation can beat the linked list's
"fast" one for thousands of elements**, which is NB-00 §1.3's constant-factor warning in its most
practical form.

It becomes a real problem when:

- **The edits are many and the list is large**, so $\Theta(n^2)$ accumulates — §1.4 measured 50
  million element moves at n = 20,000 against the linked list's 20,000 pointer writes, and 3.2
  billion against 160,000 at n = 160,000.
- **Elements are large objects moved by value** (C++ `vector` of expensive-to-move types), where
  each shift is not a cheap word copy.
- **References must stay valid.** Shifting an array invalidates indices and iterators; unlinking a
  node does not disturb anyone else's references. For a structure others point into, this is often
  the deciding factor rather than speed.

And the option people forget: if you are removing or inserting *many* elements in one pass,
**rebuild by filtering** — $\Theta(n)$, one allocation, perfect locality, and it beats both
in-place approaches. It feels wasteful and it is not.

</details>

***

### Q12. What do the classic linked-list interview problems actually test?

<details><summary>Answer</summary>

Not the structure — you will not ship one. They test three transferable things, which is why they
survive despite §3's verdict.

**1. Reasoning about aliasing.** Every bug in Part 2 is the same bug: you overwrote a pointer while
it was still your only route to something. `nxt = cur.next` before `cur.next = prev` is the whole
of §2.1. That reasoning is what you need for tree rotations (NB-08), graph adjacency edits (NB-20),
and any in-place restructuring — and it is the one skill that does not transfer from writing code
against arrays and dictionaries.

**2. Doing it in $\Theta(1)$ space.** Every classic problem has a trivial $\Theta(n)$ solution
(dump to an array, or keep a visited set) and the interesting constraint is the space bound. That
is what forces the two-pointer techniques of §2.2 and Q7, and those *do* generalise: fast/slow
pointers detect cycles in any functional graph, which is how Pollard's rho factorises integers and
how you find a duplicate in an array-as-function.

**3. Handling boundaries.** Empty, one element, two elements, deleting the head, deleting the tail.
§1.3 counted these as branches and showed sentinels erase them — and §1.5 showed they are also the
part hardest to keep consistent across two languages. Candidates fail on `[]` and `[x]` far more
often than on the algorithm.

**How to answer the meta-question if asked "would you use one?":** say no, give §3.3's three
conditions, name the LRU cache and intrusive lists as the cases that satisfy them, and cite a
number — `get(i)` being a thousand times slower says more than any adjective.

</details>

***

## Coding challenges

### Challenge 1 — an unrolled linked list

§3.1 blamed the per-node object and the lost contiguity. Attack both without giving up
$\Theta(1)$ splice.

1. Build a list whose nodes each hold an **array of $k$ elements** plus a count, rather than one
   value. Support insert, delete and traversal, keeping every node at least half full.
2. Stress-test against `list` the way §1.2 did, invariant checked after every operation.
3. Measure traversal against both a plain linked list and an array, sweeping $k$ from 1 to 256.
   Find the $k$ where most of the gap closes — you should get most of the array's locality back
   while keeping $\Theta(1)$ structural edits on node boundaries.
4. Say what you gave up. (Deleting from the middle of a node's array is $\Theta(k)$; the win is
   that $k$ is a constant you chose.)

### Challenge 2 — a skip list

The structure that gets a linked list to $\Theta(\log n)$ search, and the reason "linked lists
cannot do range queries" is not quite true.

1. Implement a skip list: a tower of linked lists where each level skips roughly twice as far,
   with levels chosen by coin flip on insert.
2. Verify search, insert and delete against a sorted reference over thousands of randomised
   sequences, asserting the level invariants after each operation.
3. Measure search against a sorted array's binary search and against a plain linked list's scan,
   and confirm the $\Theta(\log n)$ shape.
4. Compare against a balanced BST (NB-08 previews these) and say when the simpler code and easier
   concurrency are worth the constant factor. Redis sorted sets use one; find out why.

### Challenge 3 — prove §3's verdict wrong, honestly

Find a workload where a linked list beats a dynamic array on wall-clock time, and defend it.

1. Start from §3.3's bulk-removal win and construct a realistic workload around it — a scheduler,
   a free list, an event queue with cancellation.
2. Measure both, at several sizes, with the growth curves.
3. Now try to beat *both* with a third design: a **tombstone array** (mark deleted, compact when
   density falls below a threshold), or an **index-based free list** (nodes are array slots, links
   are indices rather than pointers, so you keep $\Theta(1)$ splice *and* contiguity).
4. The third design usually wins. Say why, in terms of §1.1's memory measurements and §3.1's
   locality — and note that "linked list with array indices instead of pointers" is what most
   high-performance code actually means when it says it uses a linked list.

***
# Part 5 - Practice

| # | Exercise | The technique | Difficulty |
|---|---|---|---|
| 1 | Remove duplicates from a sorted list | Single-pass relinking | ★☆☆☆☆ |
| 2 | Palindrome check in $\Theta(1)$ space | Reverse half, compare, restore | ★★★☆☆ |
| 3 | Merge $k$ sorted lists | Heap, or pairwise merge | ★★★☆☆ |
| 4 | Reverse in groups of $k$ | Reversal with bookkeeping | ★★★★☆ |
| 5 | Copy a list with random pointers | Interleave, or a node map | ★★★★☆ |
| 6 | Flatten a multilevel list | Explicit stack, or splice in place | ★★★☆☆ |
| 7 | Add two numbers as lists | Carry propagation | ★★☆☆☆ |
| 8 | Rebuild §2.4's LRU as LFU | Two-level structure | ★★★★★ |

***

### 1. Remove duplicates from a sorted list

- **Brief:** one pass; when `cur.val == cur.next.val`, splice out `cur.next`. Do **not** advance
  in that case — runs of three or more are where people slip.
- **Good result:** $\Theta(n)$, one pass, verified against a reference over thousands of random
  sorted lists including all-equal input.
- **The trap:** the variant "remove *all* nodes that have duplicates" (leaving only values that
  appear exactly once) needs a dummy head from §1.3, because the first node can be deleted. Do
  both; the second is a different problem wearing the same clothes.

### 2. Palindrome check in $\Theta(1)$ space

- **Brief:** find the middle (Q7), reverse the second half (§2.1), compare the halves, then
  **restore the list** before returning.
- **Good result:** $\Theta(n)$ time, $\Theta(1)$ space, and the list unchanged on exit — verify
  that explicitly, because it is the part everyone skips.
- **The trap:** odd versus even length, and the fact that a function that silently destroys its
  input is a bug even when it returns the right answer. If restoring is not required, say so; the
  $\Theta(n)$-space version (dump to an array, two pointers) is one line and often the better
  engineering choice.

### 3. Merge $k$ sorted lists

- **Brief:** two good answers. A **min-heap** of the $k$ current heads gives
  $\Theta(N \log k)$ (NB-09). **Pairwise merging** — merge them in rounds, halving $k$ each round —
  gives the same bound using only §2.3's two-list merge.
- **Good result:** either, implemented and verified against `sorted(concatenation)`; state the
  complexity and why the naive "merge them one at a time into an accumulator" is
  $\Theta(N k)$ instead.
- **The trap:** empty lists among the $k$, and $k = 0$. And in the heap version, ties — heapq will
  try to compare the nodes themselves once the values tie, so push `(value, index, node)`.

### 4. Reverse in groups of $k$

- **Brief:** reverse each block of $k$ nodes, leaving a trailing block of fewer than $k$ either
  reversed or not — **decide which and state it**, since both variants are asked.
- **Good result:** $\Theta(n)$ time, $\Theta(1)$ space, verified against a list-slicing reference
  for every $k$ from 1 to $n+1$.
- **The trap:** relinking the blocks to each other. You need the tail of the previous block and the
  head of the next, and you must count ahead to know whether a full group of $k$ remains before
  reversing it. A dummy head (§1.3) removes the first-block special case.

### 5. Copy a list with random pointers

Each node has `next` and an extra `random` pointer to any node or `None`. Deep-copy it.

- **Brief:** the $\Theta(n)$-space answer maps original→copy in a dict, then fixes pointers in a
  second pass. The $\Theta(1)$-extra-space answer **interleaves** the copies into the original list
  (A→A'→B→B'→…), sets `A'.random = A.random.next`, then unweaves.
- **Good result:** both, verified structurally — the copy must have identical shape with **no node
  shared** with the original. Check that explicitly by identity.
- **The trap:** the unweaving must restore the original list exactly. And `A.random` may be `None`.
  The interleaving trick is a genuinely clever use of the structure's own links as scratch space.

### 6. Flatten a multilevel doubly linked list

Nodes have `next`, `prev` and an optional `child` list to be spliced in where it appears.

- **Brief:** depth-first. When a node has a child, splice the child list between it and its
  successor and clear `child`. An explicit stack beats recursion here for the reason in Q2.
- **Good result:** $\Theta(n)$, all `prev` pointers correct afterwards, `child` cleared everywhere —
  verify by walking backwards from the tail and comparing (§1.3's invariant does exactly this).
- **The trap:** `prev` on the spliced boundaries, and a child at the very last node.

### 7. Add two numbers represented as lists

Digits in reverse order (units first); return the sum in the same form.

- **Brief:** walk both with a carry; a dummy head (§1.3) removes the first-digit case.
- **Good result:** $\Theta(\max(n,m))$, verified against integer arithmetic over thousands of random
  pairs.
- **The trap:** the final carry (99 + 1), unequal lengths, and the forward-order variant, which
  needs either reversal or a stack — and is the more interesting question.

### 8. Rebuild §2.4's LRU cache as an LFU cache

Evict the **least frequently** used entry, breaking ties by least recently used. All operations
$\Theta(1)$.

- **Brief:** a map from key to node, plus a map from **frequency to its own doubly linked list**,
  plus a running minimum frequency. A `get` moves the node from its frequency list to the next
  one — $\Theta(1)$ because you hold the node (§1.4).
- **Good result:** genuinely $\Theta(1)$ per operation, verified against an obviously-correct
  $\Theta(n)$ reference over randomised sessions, with an invariant tying all three structures
  together the way §2.4's `lru_ok` does.
- **The trap:** maintaining `min_freq` correctly when a frequency list becomes empty, and the
  tie-break. This is the hardest problem in the notebook and the best argument that the
  map-plus-linked-list pattern is worth knowing properly.

***
# Part 6 - Reading

## Start here

**1. *Introduction to Algorithms* (CLRS), chapter 10.2 — "Linked lists".**
> The reference treatment, and notable for building the doubly linked list **with a sentinel from
> the start** rather than as an optimisation — §1.3's measurement is that choice justified. Short,
> precise, and the source of the notation most other treatments borrow.

**2. [Bjarne Stroustrup: "Are lists evil?"](https://www.stroustrup.com/bs_faq2.html#list)** —
and the associated GoingNative 2012 talk. **Free.**
> Stroustrup's benchmark: generate random integers, insert each into a sorted sequence, then remove
> them in random order. `vector` beats `list` at every size he tested, *despite* `list` having the
> better complexity, because the linear scan to find the position dwarfs the $\Theta(1)$ insert.
> This is §1.4's precondition — you must already hold the node — stated as a benchmark, and it is
> the single most useful thing to have read before an argument about linked lists.

**3. [Linux `list_head`](https://github.com/torvalds/linux/blob/master/include/linux/list.h)** —
the kernel's intrusive doubly linked list. **Free.**
> Q10's answer as production code: circular, sentinel-based, intrusive, with `container_of` to
> recover the enclosing struct. Read it to see what a linked list looks like when it is genuinely
> the right structure — and note it is circular-with-sentinel, exactly §1.3's design.

## The source behind each section

| Section | Where it comes from | Free? |
|---|---|---|
| 1.1 — node cost, layout | **CPython** `Objects/typeobject.c` on `__slots__`; NB-01 §3's locality measurements | ✅ |
| 1.2–1.3 — the list, sentinels | **CLRS ch. 10.2**; **Sedgewick & Wayne, *Algorithms* §1.3** | 🔍 |
| 1.4 — $\Theta(1)$ splice | **Stroustrup**, "Are lists evil?" — the same point from the other side | ✅ |
| 2.1 — reversal, recursion depth | **CPython** `sys.setrecursionlimit` docs; NB-00 §4 | ✅ |
| 2.2 — **Floyd's algorithm** | **Knuth, TAOCP vol. 2 §3.1**, exercise 6 — where the tortoise and hare first appear in print | 🔍 |
| 2.2 — the faster alternative | **Brent**, *An improved Monte Carlo factorization algorithm*, BIT 1980 | 🔍 |
| 2.3 — merge sort on lists | **CLRS ch. 2.3**; the classic in-place list merge sort | 🔍 |
| 2.4 — LRU, map + list | **`LinkedHashMap`** source and Javadoc; Python's `OrderedDict` C implementation | ✅ |
| 3.1–3.2 — the cost of pointer chasing | **Drepper**, *What Every Programmer Should Know About Memory* (2007), §3 | ✅ |
| 3.3 — `ArrayDeque` over `LinkedList` | **`ArrayDeque` Javadoc**: "likely to be faster than `LinkedList` when used as a queue" | ✅ |
| Q10 — intrusive lists | **Linux `list.h`**; **BSD `queue.h`** | ✅ |
| Ch. 2 — skip lists | **Pugh**, *Skip lists: a probabilistic alternative to balanced trees*, CACM 1990 | 🔍 |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Stroustrup, "Are lists evil?"** It is two pages and it is this notebook's Part 3 arrived at
independently and a decade earlier. The value is not the conclusion — you now have your own
measurements for that — but the *method*: he takes a claim everyone believes on the strength of a
complexity table, designs the fairest possible experiment for it, runs it, and reports that the
table was answering a different question than the one people were asking.

Then read **Drepper §3** for why the machine behaves that way, and **CLRS 10.2** for the sentinel
treatment §1.3 measured the value of.

***
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| Traversal much slower than an array | One object per element, no contiguity (§3.1) | Use an array; if you must chain, consider unrolling (Ch. 1) |
| `for i in range(len(l)): l.get(i)` crawls | `get(i)` is $\Theta(n)$, so the loop is $\Theta(n^2)$ (§3.2) | Iterate with an iterator, or use an array |
| Infinite loop while traversing | A cycle — usually a mis-set `next` during surgery (§2.2) | Floyd's detection; assert the invariant after every edit (§1.2) |
| Lost the rest of the list mid-edit | Overwrote `cur.next` before saving it (§2.1) | Save `nxt` **first**; three-pointer discipline |
| `RecursionError` / `StackOverflowError` | Recursion depth $\Theta(n)$ (§2.1, Q2) | Rewrite iteratively; do not raise the limit |
| Off-by-one at head or tail | Boundary branches got it wrong (§1.3) | A sentinel node — 4 structural branches become 0 |
| Stale `tail` after deleting the last node | Forgot to update it (§1.2's invariant catches this) | Assert the tail invariant after every operation |
| Node deleted but still reachable | Only one side of a doubly linked pair was updated | `unlink` both directions; verify forward == reverse(backward) |
| Memory much higher than expected | 48 B/node with `__slots__`, **136 B without** (§1.1) | Add `__slots__`; or do not use nodes |
| LRU cache returns stale or missing entries | Map and list drifted apart (§2.4) | An invariant that compares **both** structures |
| Queue built on `ArrayList` is quadratic | `remove(0)` shifts everything (§3.3) | `ArrayDeque` / `collections.deque`, not `LinkedList` |
| Cycle detector reports the wrong entry node | Step sizes where $(b-a) \nmid a$ (§2.2) | Use 1 and 2 |
| Deleting many elements is slow either way | Wrong approach entirely (§1.4) | Rebuild by filtering — $\Theta(n)$, one pass |

## Checklist for linked-list code

- [ ] Does this need to be a linked list at all — do §3.3's **three** conditions hold?
- [ ] Is there a **sentinel**, or are boundary branches being hand-written (§1.3)?
- [ ] Is every pointer read saved before the pointer is overwritten (§2.1)?
- [ ] Is any recursion depth $\Theta(n)$ (§2.1)?
- [ ] For a doubly linked list, does every edit update **both** directions (§1.3)?
- [ ] Is the invariant asserted after every mutation, with a **step budget** so a cycle reports
      rather than hangs (§1.2)?
- [ ] Do nodes have `__slots__`, or is 3× the memory being paid for nothing (§1.1)?
- [ ] Is anything indexing into it in a loop (§3.2)?
- [ ] If it is a queue, is it `deque`/`ArrayDeque` rather than a linked list (§3.3)?
- [ ] Was the complexity **measured** — or the operations **counted** where timing could not
      settle it (§1.4)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `stacks_queues_zero_to_hero.ipynb` | §3.3's `ArrayDeque` result in full: the circular buffer that beats a linked list at its own game |
| `trees_zero_to_hero.ipynb` | Nodes and pointers with two children — §2.1's aliasing discipline, harder |
| [`hashing_zero_to_hero.ipynb`](hashing_zero_to_hero.ipynb) | §1.3's chains, and the map half of §2.4's LRU cache |
| [`arrays_zero_to_hero.ipynb`](arrays_zero_to_hero.ipynb) | The structure §3 keeps telling you to use instead |

See [`README.md`](README.md) for the full roster and reading order.